In [1]:
# Individual Blob Kinetics Analysis

## Goal
This notebook is used to:
- select isolated blobs in each frame
- link blobs across consecutive frames
- keep tracks with at least 2 consecutive frames
- calculate individual blob kinetics:
  - mean_r intensity per time
  - intensity gradient (dI/dt)
- calculate field-level summary per timepoint:
  - mean mean_r
  - mean dI/dt
- export results to Excel
- export plots

## Definitions
- **Isolated blob**: a blob that is sufficiently separated from other blobs in the same frame and does not overlap with neighbors.
- **Continuous track**: a blob track that exists in at least 2 consecutive frames with no skipped frames.
- **No kinetic threshold is applied at the selection stage**:
  all isolated blobs with continuous tracks are kept for downstream analysis.

SyntaxError: invalid syntax (3433759274.py, line 4)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from itertools import combinations
import math
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [ ]:
CONFIG = {
    # required column names
    "frame_col": "frame_index_kept",
    "x_col": "x",
    "y_col": "y",
    "radius_col": "radius",
    "intensity_col": "mean_r",

    # timing
    "frame_interval_min": 30,
    "min_consecutive_frames": 2,

    # isolation criteria
    # primary criterion: nearest-neighbor distance
    "min_nn_distance": 15.0,          # tune later if needed

    # overlap logic (used for QC; optional for final isolation rule)
    "overlap_margin": 0.0,
    "overlap_factor": 0.6,           # < 1.0 means stricter "true overlap" only
    "use_overlap_for_isolation": False,

    # tracking / linking
    "max_link_distance": 10.0,       # tune later
    "max_radius_delta": None,        # optional, e.g. 3.0

    # plotting
    "max_time_min": 240,
    "save_plots": True,
    "show_plots": True,
}

In [ ]:
INPUT_DIR = Path("inputs/individual_blob_kinetics")
OUTPUT_DIR = Path("outputs/individual_blob_kinetics")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

input_files = sorted(INPUT_DIR.glob("*.xlsx"))

print(f"Found {len(input_files)} input files")
for f in input_files:
    print("-", f.name)

In [ ]:
def sanity_check_blob_file(input_file, config):
    input_file = Path(input_file)
    df = pd.read_excel(input_file)

    required_cols = [
        config["frame_col"],
        config["x_col"],
        config["y_col"],
        config["radius_col"],
        config["intensity_col"],
    ]

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{input_file.name} is missing required columns: {missing}")

    print(f"OK: {input_file.name} | rows={len(df)} | cols={len(df.columns)}")
    return df


def sanity_check_all_input_files(input_files, config):
    checked = []
    for f in input_files:
        df = sanity_check_blob_file(f, config)
        checked.append({
            "file": Path(f).name,
            "n_rows": len(df),
            "n_cols": len(df.columns),
            "status": "ok"
        })
    return pd.DataFrame(checked)

In [ ]:
if len(input_files) > 0:
    check_df = sanity_check_all_input_files(input_files, CONFIG)
    display(check_df)
else:
    print("No input files found yet.")

In [ ]:
print("Current working directory:")
print(Path.cwd())

print("\nDoes INPUT_DIR exist?")
print(INPUT_DIR.exists(), INPUT_DIR)

if INPUT_DIR.exists():
    print("\nFiles inside INPUT_DIR:")
    for p in INPUT_DIR.iterdir():
        print("-", p.name)

In [ ]:
def euclidean_distance(x1, y1, x2, y2):
    return float(np.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2))


def compute_frame_isolation_metrics(frame_df, config):
    """
    Compute isolation metrics for blobs within a single frame.

    Adds:
    - nn_distance: distance to nearest neighbor center
    - min_center_distance: same as nn_distance (explicit name for readability)
    - is_overlapping: True if strongly overlaps with any other blob in same frame
    - overlap_neighbor_count: number of overlapping neighbors
    """
    frame_df = frame_df.copy().reset_index(drop=False).rename(columns={"index": "_orig_index"})

    x_col = config["x_col"]
    y_col = config["y_col"]
    radius_col = config["radius_col"]

    overlap_margin = config.get("overlap_margin", 0.0)
    overlap_factor = config.get("overlap_factor", 1.0)

    n = len(frame_df)

    if n == 1:
        frame_df["nn_distance"] = np.nan
        frame_df["min_center_distance"] = np.nan
        frame_df["is_overlapping"] = False
        frame_df["overlap_neighbor_count"] = 0
        return frame_df

    nn_distances = []
    overlap_flags = []
    overlap_counts = []

    for i in range(n):
        xi = frame_df.loc[i, x_col]
        yi = frame_df.loc[i, y_col]
        ri = frame_df.loc[i, radius_col]

        distances = []
        overlap_count = 0

        for j in range(n):
            if i == j:
                continue

            xj = frame_df.loc[j, x_col]
            yj = frame_df.loc[j, y_col]
            rj = frame_df.loc[j, radius_col]

            d = euclidean_distance(xi, yi, xj, yj)
            distances.append(d)

            # softer overlap definition:
            # counts as overlap only when centers are closer than
            # overlap_factor * (ri + rj) + margin
            if d <= (overlap_factor * (ri + rj) + overlap_margin):
                overlap_count += 1

        nn_d = min(distances) if len(distances) > 0 else np.nan
        nn_distances.append(nn_d)
        overlap_flags.append(overlap_count > 0)
        overlap_counts.append(overlap_count)

    frame_df["nn_distance"] = nn_distances
    frame_df["min_center_distance"] = nn_distances
    frame_df["is_overlapping"] = overlap_flags
    frame_df["overlap_neighbor_count"] = overlap_counts

    return frame_df


def compute_isolation_metrics_all_frames(df, config):
    """
    Compute isolation metrics frame by frame across the full dataframe.
    """
    frame_col = config["frame_col"]

    out = []
    for frame_value, sub_df in df.groupby(frame_col, sort=True):
        sub_out = compute_frame_isolation_metrics(sub_df, config)
        out.append(sub_out)

    result = pd.concat(out, axis=0, ignore_index=True)

    if "_orig_index" in result.columns:
        result = result.drop(columns=["_orig_index"])

    return result

In [ ]:
def mark_isolated_blobs(df, config):
    """
    Mark blobs as isolated based on nearest-neighbor distance.
    Optionally also require non-overlap depending on config.

    Default recommended behavior:
    - use nearest-neighbor distance as the main rule
    - keep overlap only as QC unless explicitly enabled
    """
    df = df.copy()

    min_nn_distance = config["min_nn_distance"]
    use_overlap = config.get("use_overlap_for_isolation", False)

    # If nn_distance is NaN (e.g. only one blob in frame), treat as acceptable
    nn_ok = df["nn_distance"].isna() | (df["nn_distance"] >= min_nn_distance)

    if use_overlap:
        overlap_ok = ~df["is_overlapping"]
        df["is_isolated"] = nn_ok & overlap_ok
    else:
        df["is_isolated"] = nn_ok

    return df


def build_isolation_qc_summary(df, config):
    """
    Build a simple QC summary for isolation results.
    """
    total_blobs = len(df)
    isolated_blobs = int(df["is_isolated"].sum()) if "is_isolated" in df.columns else 0
    overlapping_blobs = int(df["is_overlapping"].sum()) if "is_overlapping" in df.columns else 0

    summary = pd.DataFrame([
        {"metric": "total_blobs", "value": total_blobs},
        {"metric": "isolated_blobs", "value": isolated_blobs},
        {"metric": "non_isolated_blobs", "value": total_blobs - isolated_blobs},
        {"metric": "overlapping_blobs", "value": overlapping_blobs},
        {
            "metric": "fraction_isolated",
            "value": isolated_blobs / total_blobs if total_blobs > 0 else np.nan,
        },
    ])

    return summary

In [ ]:
if len(input_files) > 0:
    test_file = input_files[0]
    print("Testing isolation on:", test_file.name)

    raw_df = pd.read_excel(test_file)

    iso_df = compute_isolation_metrics_all_frames(raw_df, CONFIG)
    iso_df = mark_isolated_blobs(iso_df, CONFIG)

    qc_df = build_isolation_qc_summary(iso_df, CONFIG)

    print("\nIsolation QC summary:")
    display(qc_df)

    print("\nPreview of isolation result:")
    preview_cols = [
        CONFIG["frame_col"],
        CONFIG["x_col"],
        CONFIG["y_col"],
        CONFIG["radius_col"],
        CONFIG["intensity_col"],
        "nn_distance",
        "is_overlapping",
        "overlap_neighbor_count",
        "is_isolated",
    ]
    preview_cols = [c for c in preview_cols if c in iso_df.columns]

    display(iso_df[preview_cols].head(20))

else:
    print("No input files found yet. Add at least one .xlsx file, then rerun this cell.")

In [ ]:
from pathlib import Path

print("Current working directory:")
print(Path.cwd())

print("\nINPUT_DIR as written in notebook:")
print(INPUT_DIR)

print("\nResolved INPUT_DIR:")
print(INPUT_DIR.resolve())

print("\nDoes INPUT_DIR exist?")
print(INPUT_DIR.exists())

if INPUT_DIR.exists():
    print("\nContents of INPUT_DIR:")
    for p in INPUT_DIR.iterdir():
        print("-", p.name)
else:
    print("\nINPUT_DIR does not exist from the notebook's current working directory.")

In [ ]:
input_files = sorted(INPUT_DIR.glob("*.xlsx"))

print(f"Found {len(input_files)} xlsx files")
for f in input_files:
    print("-", f.name)

In [ ]:
if len(input_files) > 0:
    check_df = sanity_check_all_input_files(input_files, CONFIG)
    display(check_df)
else:
    print("No input files found yet.")

In [ ]:
if len(input_files) > 0:
    test_file = input_files[0]
    raw_df = pd.read_excel(test_file)

    iso_df = compute_isolation_metrics_all_frames(raw_df, CONFIG)

    print(iso_df["nn_distance"].describe())

    plt.figure(figsize=(6, 4))
    plt.hist(iso_df["nn_distance"].dropna(), bins=20, edgecolor="black")
    plt.xlabel("Nearest neighbor distance")
    plt.ylabel("Count")
    plt.title("Distribution of nn_distance")
    plt.grid(alpha=0.2)
    plt.show()

In [ ]:
if len(input_files) > 0:
    test_file = input_files[0]
    raw_df = pd.read_excel(test_file)

    iso_df = compute_isolation_metrics_all_frames(raw_df, CONFIG)

    thresholds = [4, 6, 8, 10, 12, 15]

    rows = []
    for th in thresholds:
        temp_config = CONFIG.copy()
        temp_config["min_nn_distance"] = th

        temp_df = mark_isolated_blobs(iso_df, temp_config)

        rows.append({
            "min_nn_distance": th,
            "isolated_blobs": int(temp_df["is_isolated"].sum()),
            "total_blobs": len(temp_df),
            "fraction_isolated": temp_df["is_isolated"].mean()
        })

    threshold_test_df = pd.DataFrame(rows)
    display(threshold_test_df)
else:
    print("No input files found.")

In [ ]:
def link_two_consecutive_frames(df_t, df_t1, config):
    """
    Link isolated blobs from frame t to frame t+1 using nearest-neighbor matching.

    Returns a DataFrame with one row per accepted link:
    - row_index_t
    - row_index_t1
    - link_distance
    """
    x_col = config["x_col"]
    y_col = config["y_col"]
    radius_col = config["radius_col"]

    max_link_distance = config["max_link_distance"]
    max_radius_delta = config.get("max_radius_delta", None)

    if len(df_t) == 0 or len(df_t1) == 0:
        return pd.DataFrame(columns=["row_index_t", "row_index_t1", "link_distance"])

    candidates = []

    for i, row_t in df_t.iterrows():
        xt, yt = row_t[x_col], row_t[y_col]
        rt = row_t[radius_col]

        for j, row_t1 in df_t1.iterrows():
            x1, y1 = row_t1[x_col], row_t1[y_col]
            r1 = row_t1[radius_col]

            d = euclidean_distance(xt, yt, x1, y1)

            if d > max_link_distance:
                continue

            if max_radius_delta is not None:
                if abs(rt - r1) > max_radius_delta:
                    continue

            candidates.append({
                "row_index_t": i,
                "row_index_t1": j,
                "link_distance": d,
            })

    if len(candidates) == 0:
        return pd.DataFrame(columns=["row_index_t", "row_index_t1", "link_distance"])

    cand_df = pd.DataFrame(candidates).sort_values("link_distance").reset_index(drop=True)

    # Greedy one-to-one matching: smallest distances first
    used_t = set()
    used_t1 = set()
    accepted = []

    for _, row in cand_df.iterrows():
        i = row["row_index_t"]
        j = row["row_index_t1"]

        if i in used_t or j in used_t1:
            continue

        accepted.append(row.to_dict())
        used_t.add(i)
        used_t1.add(j)

    accepted_df = pd.DataFrame(accepted)
    return accepted_df

In [ ]:
def build_blob_tracks(df, config):
    """
    Build blob tracks across consecutive frames using isolated blobs only.

    Steps:
    1. Keep only is_isolated == True
    2. For each consecutive frame pair t -> t+1, link blobs
    3. Assign / propagate track_id

    Returns:
    - full dataframe with new columns:
        track_id
        linked_from_prev
        linked_to_next
    """
    df = df.copy().reset_index(drop=True)

    frame_col = config["frame_col"]

    # initialize tracking columns
    df["track_id"] = np.nan
    df["linked_from_prev"] = False
    df["linked_to_next"] = False

    # only isolated blobs are eligible for linking
    eligible_mask = df["is_isolated"].fillna(False)
    eligible_df = df.loc[eligible_mask].copy()

    if len(eligible_df) == 0:
        return df

    frames = sorted(eligible_df[frame_col].dropna().unique())

    next_track_id = 1

    # Ensure any eligible blob in first encountered frame can receive a track if linked or standalone
    for frame_idx, t in enumerate(frames[:-1]):
        t1 = frames[frame_idx + 1]

        # Only consecutive frames are linked
        if t1 != t + 1:
            continue

        df_t = eligible_df.loc[eligible_df[frame_col] == t].copy()
        df_t1 = eligible_df.loc[eligible_df[frame_col] == t1].copy()

        links_df = link_two_consecutive_frames(df_t, df_t1, config)

        # Assign new track IDs to untracked parents when needed, then propagate to children
        for _, link_row in links_df.iterrows():
            i = int(link_row["row_index_t"])
            j = int(link_row["row_index_t1"])

            # parent track
            parent_track = df.loc[i, "track_id"]

            if pd.isna(parent_track):
                parent_track = next_track_id
                df.loc[i, "track_id"] = parent_track
                next_track_id += 1

            df.loc[j, "track_id"] = parent_track
            df.loc[i, "linked_to_next"] = True
            df.loc[j, "linked_from_prev"] = True

    # Give standalone track IDs to any isolated blobs that were never linked
    remaining_mask = df["is_isolated"].fillna(False) & df["track_id"].isna()
    remaining_indices = df.index[remaining_mask].tolist()

    for idx in remaining_indices:
        df.loc[idx, "track_id"] = next_track_id
        next_track_id += 1

    df["track_id"] = df["track_id"].astype("Int64")
    return df

In [ ]:
def is_consecutive_track(track_df, frame_col):
    """
    Returns True if the frames in this track are strictly consecutive.
    Example:
    [2,3] -> True
    [2,3,4,5] -> True
    [2,4] -> False
    """
    frames = sorted(track_df[frame_col].dropna().astype(int).unique())

    if len(frames) < 2:
        return False

    diffs = np.diff(frames)
    return np.all(diffs == 1)


def select_continuous_tracks(df, config):
    """
    Keep only tracks that:
    - belong to isolated blobs
    - have at least min_consecutive_frames
    - are strictly consecutive in time
    """
    df = df.copy()
    frame_col = config["frame_col"]
    min_consecutive_frames = config["min_consecutive_frames"]

    df["track_length"] = pd.NA
    df["is_continuous_track"] = False
    df["selected_for_kinetics"] = False

    valid_track_ids = []

    for track_id, sub_df in df.groupby("track_id", dropna=True):
        if len(sub_df) == 0:
            continue

        frames = sorted(sub_df[frame_col].dropna().astype(int).unique())
        track_length = len(frames)
        consecutive = is_consecutive_track(sub_df, frame_col)

        df.loc[sub_df.index, "track_length"] = track_length
        df.loc[sub_df.index, "is_continuous_track"] = consecutive

        if consecutive and track_length >= min_consecutive_frames:
            valid_track_ids.append(track_id)

    valid_track_ids = pd.Series(valid_track_ids, dtype="Int64").dropna().tolist()

    selected_mask = (
        df["track_id"].isin(valid_track_ids) &
        df["is_isolated"].fillna(False)
    )

    df.loc[selected_mask, "selected_for_kinetics"] = True

    selected_df = df.loc[df["selected_for_kinetics"]].copy()

    return df, selected_df


def build_tracking_qc_summary(df, selected_df):
    """
    Simple QC summary after tracking and continuous-track selection.
    """
    total_rows = len(df)
    isolated_rows = int(df["is_isolated"].sum()) if "is_isolated" in df.columns else 0
    linked_rows = int((df["linked_from_prev"] | df["linked_to_next"]).sum()) if "linked_from_prev" in df.columns else 0

    n_tracks_total = df["track_id"].dropna().nunique() if "track_id" in df.columns else 0
    n_selected_tracks = selected_df["track_id"].dropna().nunique() if len(selected_df) > 0 else 0

    summary = pd.DataFrame([
        {"metric": "total_rows", "value": total_rows},
        {"metric": "isolated_rows", "value": isolated_rows},
        {"metric": "linked_rows", "value": linked_rows},
        {"metric": "total_tracks", "value": n_tracks_total},
        {"metric": "selected_rows", "value": len(selected_df)},
        {"metric": "selected_tracks", "value": n_selected_tracks},
    ])

    return summary

In [ ]:
if len(input_files) > 0:
    test_file = input_files[0]
    print("Testing tracking on:", test_file.name)

    raw_df = pd.read_excel(test_file)

    # isolation
    iso_df = compute_isolation_metrics_all_frames(raw_df, CONFIG)
    iso_df = mark_isolated_blobs(iso_df, CONFIG)

    # tracking
    tracked_df = build_blob_tracks(iso_df, CONFIG)

    # selection
    tracked_df, selected_df = select_continuous_tracks(tracked_df, CONFIG)

    # QC
    tracking_qc_df = build_tracking_qc_summary(tracked_df, selected_df)

    print("\nTracking QC summary:")
    display(tracking_qc_df)

    print("\nTracked dataframe preview:")
    preview_cols = [
        CONFIG["frame_col"],
        CONFIG["x_col"],
        CONFIG["y_col"],
        CONFIG["radius_col"],
        CONFIG["intensity_col"],
        "nn_distance",
        "is_isolated",
        "track_id",
        "linked_from_prev",
        "linked_to_next",
        "track_length",
        "is_continuous_track",
        "selected_for_kinetics",
    ]
    preview_cols = [c for c in preview_cols if c in tracked_df.columns]
    display(tracked_df[preview_cols].head(30))

    print("\nSelected rows preview:")
    if len(selected_df) > 0:
        display(selected_df[preview_cols].head(30))
    else:
        print("No continuous tracks selected.")
else:
    print("No input files found yet.")

In [ ]:
def compute_individual_kinetics_manuscript_style(selected_df, config, stable_eps=1e-6):
    """
    Compute kinetics in manuscript/software-style with two outputs:

    1) selected_point_df
       - one row per tracked blob per timepoint
       - used for intensity-time trajectory

    2) selected_interval_df
       - one row per consecutive interval within the same track
       - used for dI/dt analysis

    Definitions:
        dI/dt = (I_t1 - I_t) / (time_t1 - time_t)

    Time handling in this version:
        - point time is stored in both minutes and hours
        - gradient is computed per hour
        - gradient is plotted at interval midpoint in hours
    """
    if len(selected_df) == 0:
        empty_point_df = selected_df.copy()

        empty_interval_df = pd.DataFrame(columns=[
            "track_id",
            "frame_t",
            "frame_t1",
            "time_t_min",
            "time_t1_min",
            "time_mid_min",
            "time_t_hr",
            "time_t1_hr",
            "time_mid_hr",
            "delta_time_hr",
            "I_t",
            "I_t1",
            "delta_intensity",
            "dI_dt",
            "phase_class",
        ])
        return empty_point_df, empty_interval_df

    frame_col = config["frame_col"]
    intensity_col = config["intensity_col"]
    frame_interval_min = config["frame_interval_min"]

    # ----------------------------
    # point-level table
    # ----------------------------
    selected_point_df = selected_df.copy()
    selected_point_df = selected_point_df.sort_values(["track_id", frame_col]).reset_index(drop=True)

    selected_point_df["time_min"] = selected_point_df[frame_col] * frame_interval_min
    selected_point_df["time_hr"] = selected_point_df["time_min"] / 60.0

    # ----------------------------
    # interval-level table
    # ----------------------------
    interval_rows = []

    for track_id, sub_df in selected_point_df.groupby("track_id", dropna=True):
        sub_df = sub_df.sort_values(frame_col).reset_index(drop=True)

        if len(sub_df) < 2:
            continue

        for i in range(len(sub_df) - 1):
            row_t = sub_df.iloc[i]
            row_t1 = sub_df.iloc[i + 1]

            frame_t = row_t[frame_col]
            frame_t1 = row_t1[frame_col]

            time_t_min = row_t["time_min"]
            time_t1_min = row_t1["time_min"]

            time_t_hr = row_t["time_hr"]
            time_t1_hr = row_t1["time_hr"]

            I_t = row_t[intensity_col]
            I_t1 = row_t1[intensity_col]

            delta_time_hr = time_t1_hr - time_t_hr
            delta_intensity = I_t1 - I_t

            if pd.isna(delta_time_hr) or delta_time_hr == 0:
                dI_dt = np.nan
            else:
                dI_dt = delta_intensity / delta_time_hr

            if pd.isna(dI_dt):
                phase_class = "undefined"
            elif dI_dt > stable_eps:
                phase_class = "brightening"
            elif dI_dt < -stable_eps:
                phase_class = "dimming"
            else:
                phase_class = "stable"

            interval_rows.append({
                "track_id": track_id,
                "frame_t": frame_t,
                "frame_t1": frame_t1,
                "time_t_min": time_t_min,
                "time_t1_min": time_t1_min,
                "time_mid_min": (time_t_min + time_t1_min) / 2.0,
                "time_t_hr": time_t_hr,
                "time_t1_hr": time_t1_hr,
                "time_mid_hr": (time_t_hr + time_t1_hr) / 2.0,
                "delta_time_hr": delta_time_hr,
                "I_t": I_t,
                "I_t1": I_t1,
                "delta_intensity": delta_intensity,
                "dI_dt": dI_dt,
                "phase_class": phase_class,
            })

    selected_interval_df = pd.DataFrame(interval_rows)

    return selected_point_df, selected_interval_df

In [ ]:
def build_track_level_summary(selected_point_df, selected_interval_df, config):
    """
    Build one-row-per-track summary using:
    - point-level intensity table
    - interval-level gradient table
    """
    if len(selected_point_df) == 0:
        return pd.DataFrame()

    frame_col = config["frame_col"]
    intensity_col = config["intensity_col"]

    rows = []

    for track_id, sub_point_df in selected_point_df.groupby("track_id", dropna=True):
        sub_point_df = sub_point_df.sort_values(frame_col).copy()
        sub_interval_df = selected_interval_df.loc[
            selected_interval_df["track_id"] == track_id
        ].copy()

        frames = sub_point_df[frame_col].dropna().astype(int).tolist()
        times_min = sub_point_df["time_min"].dropna().tolist()
        times_hr = sub_point_df["time_hr"].dropna().tolist()
        intensities = sub_point_df[intensity_col].dropna().tolist()
        gradients = sub_interval_df["dI_dt"].dropna().tolist()

        row = {
            "track_id": track_id,
            "track_length_points": len(frames),
            "track_length_intervals": len(sub_interval_df),
            "first_frame": min(frames) if len(frames) > 0 else np.nan,
            "last_frame": max(frames) if len(frames) > 0 else np.nan,
            "first_time_min": min(times_min) if len(times_min) > 0 else np.nan,
            "last_time_min": max(times_min) if len(times_min) > 0 else np.nan,
            "first_time_hr": min(times_hr) if len(times_hr) > 0 else np.nan,
            "last_time_hr": max(times_hr) if len(times_hr) > 0 else np.nan,
            "start_mean_r": intensities[0] if len(intensities) > 0 else np.nan,
            "end_mean_r": intensities[-1] if len(intensities) > 0 else np.nan,
            "min_mean_r": np.min(intensities) if len(intensities) > 0 else np.nan,
            "max_mean_r": np.max(intensities) if len(intensities) > 0 else np.nan,
            "mean_mean_r": np.mean(intensities) if len(intensities) > 0 else np.nan,
            "mean_dI_dt": np.mean(gradients) if len(gradients) > 0 else np.nan,
            "max_dI_dt": np.max(gradients) if len(gradients) > 0 else np.nan,
            "min_dI_dt": np.min(gradients) if len(gradients) > 0 else np.nan,
            "n_brightening_intervals": int((sub_interval_df["phase_class"] == "brightening").sum()) if len(sub_interval_df) > 0 else 0,
            "n_dimming_intervals": int((sub_interval_df["phase_class"] == "dimming").sum()) if len(sub_interval_df) > 0 else 0,
            "n_stable_intervals": int((sub_interval_df["phase_class"] == "stable").sum()) if len(sub_interval_df) > 0 else 0,
        }

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
if len(input_files) > 0:
    test_file = input_files[0]
    print("Testing manuscript-style kinetics on:", test_file.name)

    raw_df = pd.read_excel(test_file)

    # isolation
    iso_df = compute_isolation_metrics_all_frames(raw_df, CONFIG)
    iso_df = mark_isolated_blobs(iso_df, CONFIG)

    # tracking
    tracked_df = build_blob_tracks(iso_df, CONFIG)
    tracked_df, selected_df = select_continuous_tracks(tracked_df, CONFIG)

    # manuscript-style kinetics
    selected_point_df, selected_interval_df = compute_individual_kinetics_manuscript_style(
        selected_df, CONFIG
    )

    print("\nPoint-level table preview:")
    display(
        selected_point_df[[
            "track_id",
            CONFIG["frame_col"],
            "time_min",
            CONFIG["intensity_col"]
        ]].head(20)
    )

    print("\nInterval-level table preview:")
    if len(selected_interval_df) > 0:
        display(selected_interval_df.head(20))
    else:
        print("No interval data available.")
else:
    print("No input files found yet.")

In [ ]:
def build_timepoint_summary(selected_point_df, selected_interval_df, config):
    """
    Build two summary tables:

    1) point_summary_df
       - summary of mean_r at each point time (time_hr)

    2) interval_summary_df
       - summary of dI/dt at each interval midpoint time (time_mid_hr)
    """
    intensity_col = config["intensity_col"]

    # ----------------------------
    # point-level summary
    # ----------------------------
    if len(selected_point_df) > 0:
        point_summary_df = (
            selected_point_df
            .groupby(["time_min", "time_hr"], dropna=False)
            .agg(
                n_selected_blobs=("track_id", "count"),
                mean_mean_r=(intensity_col, "mean"),
                std_mean_r=(intensity_col, "std"),
                min_mean_r=(intensity_col, "min"),
                max_mean_r=(intensity_col, "max"),
            )
            .reset_index()
            .sort_values(["time_hr"])
            .reset_index(drop=True)
        )
    else:
        point_summary_df = pd.DataFrame(columns=[
            "time_min", "time_hr",
            "n_selected_blobs",
            "mean_mean_r", "std_mean_r",
            "min_mean_r", "max_mean_r"
        ])

    # ----------------------------
    # interval-level summary
    # ----------------------------
    if len(selected_interval_df) > 0:
        interval_summary_df = (
            selected_interval_df
            .groupby(["time_mid_min", "time_mid_hr"], dropna=False)
            .agg(
                n_intervals=("track_id", "count"),
                mean_dI_dt=("dI_dt", "mean"),
                std_dI_dt=("dI_dt", "std"),
                min_dI_dt=("dI_dt", "min"),
                max_dI_dt=("dI_dt", "max"),
            )
            .reset_index()
            .sort_values(["time_mid_hr"])
            .reset_index(drop=True)
        )
    else:
        interval_summary_df = pd.DataFrame(columns=[
            "time_mid_min", "time_mid_hr",
            "n_intervals",
            "mean_dI_dt", "std_dI_dt",
            "min_dI_dt", "max_dI_dt"
        ])

    return point_summary_df, interval_summary_df

In [ ]:
def build_direction_summary_from_intervals(selected_interval_df, stable_eps=1e-6):
    """
    Build direction summary from interval-level gradients.

    Classification:
    - brightening: dI_dt > stable_eps
    - dimming: dI_dt < -stable_eps
    - stable: abs(dI_dt) <= stable_eps
    - undefined: NaN
    """
    if len(selected_interval_df) == 0:
        return pd.DataFrame(columns=[
            "time_mid_min",
            "time_mid_hr",
            "n_intervals",
            "n_brightening",
            "n_dimming",
            "n_stable",
            "n_undefined",
            "fraction_brightening",
            "fraction_dimming",
            "fraction_stable",
        ])

    df = selected_interval_df.copy()

    def classify_direction(x):
        if pd.isna(x):
            return "undefined"
        elif x > stable_eps:
            return "brightening"
        elif x < -stable_eps:
            return "dimming"
        else:
            return "stable"

    df["phase_class"] = df["dI_dt"].apply(classify_direction)

    direction_df = (
        df.groupby(["time_mid_min", "time_mid_hr"], dropna=False)
        .agg(
            n_intervals=("track_id", "count"),
            n_brightening=("phase_class", lambda s: (s == "brightening").sum()),
            n_dimming=("phase_class", lambda s: (s == "dimming").sum()),
            n_stable=("phase_class", lambda s: (s == "stable").sum()),
            n_undefined=("phase_class", lambda s: (s == "undefined").sum()),
        )
        .reset_index()
        .sort_values(["time_mid_hr"])
        .reset_index(drop=True)
    )

    direction_df["fraction_brightening"] = direction_df["n_brightening"] / direction_df["n_intervals"]
    direction_df["fraction_dimming"] = direction_df["n_dimming"] / direction_df["n_intervals"]
    direction_df["fraction_stable"] = direction_df["n_stable"] / direction_df["n_intervals"]

    return direction_df

In [ ]:
if len(input_files) > 0:
    test_file = input_files[0]
    print("Testing kinetics on:", test_file.name)

    raw_df = pd.read_excel(test_file)

    # isolation
    iso_df = compute_isolation_metrics_all_frames(raw_df, CONFIG)
    iso_df = mark_isolated_blobs(iso_df, CONFIG)

    # tracking
    tracked_df = build_blob_tracks(iso_df, CONFIG)
    tracked_df, selected_df = select_continuous_tracks(tracked_df, CONFIG)

    # kinetics
    selected_df = compute_individual_kinetics(selected_df, CONFIG)
    track_summary_df = build_track_level_summary(selected_df, CONFIG)
    timepoint_summary_df = build_timepoint_summary(selected_df, CONFIG)
    direction_summary_df = build_direction_summary(selected_df, CONFIG)

    print("\nSelected blob kinetics preview:")
    display(selected_df.head(20))

    print("\nTrack-level summary preview:")
    display(track_summary_df.head(20))

    print("\nTimepoint summary preview:")
    display(timepoint_summary_df.head(20))

    print("\nDirection summary preview:")
    display(direction_summary_df.head(20))

else:
    print("No input files found yet.")

In [ ]:
def build_selection_qc_table(raw_df, iso_df, tracked_df, selected_point_df, selected_interval_df):
    """
    Build a compact QC table summarizing selection steps.
    """
    total_raw = len(raw_df)
    total_isolated = int(iso_df["is_isolated"].sum()) if "is_isolated" in iso_df.columns else 0
    total_tracks = tracked_df["track_id"].dropna().nunique() if "track_id" in tracked_df.columns else 0
    total_selected_points = len(selected_point_df)
    total_selected_tracks = selected_point_df["track_id"].dropna().nunique() if len(selected_point_df) > 0 else 0
    total_selected_intervals = len(selected_interval_df)

    qc_df = pd.DataFrame([
        {"step": "raw_input", "metric": "total_rows", "value": total_raw},
        {"step": "isolation", "metric": "isolated_rows", "value": total_isolated},
        {"step": "tracking", "metric": "total_tracks", "value": total_tracks},
        {"step": "selection", "metric": "selected_points", "value": total_selected_points},
        {"step": "selection", "metric": "selected_tracks", "value": total_selected_tracks},
        {"step": "kinetics", "metric": "selected_intervals", "value": total_selected_intervals},
    ])

    return qc_df


def export_blob_kinetics_excel(
    raw_df,
    iso_df,
    tracked_df,
    selected_point_df,
    selected_interval_df,
    track_summary_df,
    point_summary_df,
    interval_summary_df,
    direction_summary_df,
    selection_qc_df,
    output_path
):
    """
    Export all result tables to a multi-sheet Excel file.
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        raw_df.to_excel(writer, sheet_name="all_blobs_raw", index=False)
        iso_df.to_excel(writer, sheet_name="isolation_metrics", index=False)
        tracked_df.to_excel(writer, sheet_name="tracked_blobs", index=False)
        selected_point_df.to_excel(writer, sheet_name="selected_blob_points", index=False)
        selected_interval_df.to_excel(writer, sheet_name="selected_blob_intervals", index=False)
        track_summary_df.to_excel(writer, sheet_name="track_level_summary", index=False)
        point_summary_df.to_excel(writer, sheet_name="point_summary", index=False)
        interval_summary_df.to_excel(writer, sheet_name="interval_summary", index=False)
        direction_summary_df.to_excel(writer, sheet_name="direction_summary", index=False)
        selection_qc_df.to_excel(writer, sheet_name="selection_qc", index=False)

    print(f"Saved Excel: {output_path}")

In [ ]:
def plot_individual_intensity(selected_point_df, config, output_png=None, show_plot=True):
    """
    Plot mean_r vs time for each selected track.
    Uses point-level table.
    """
    if len(selected_point_df) == 0:
        print("No selected point data to plot: individual intensity.")
        return

    intensity_col = config["intensity_col"]

    fig, ax = plt.subplots(figsize=(7, 5))

    for track_id, sub_df in selected_point_df.groupby("track_id", dropna=True):
        sub_df = sub_df.sort_values("time_hr")
        ax.plot(
            sub_df["time_hr"],
            sub_df[intensity_col],
            marker="o",
            linewidth=1,
            alpha=0.7,
            label=f"track {track_id}"
        )

    ax.set_xlabel("Time (hr)")
    ax.set_ylabel(intensity_col)
    ax.set_title("Individual blob mean_r over time")
    ax.grid(alpha=0.2)

    if selected_point_df["track_id"].nunique() <= 12:
        ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")

    plt.tight_layout()

    if output_png is not None:
        plt.savefig(output_png, dpi=300, bbox_inches="tight")
        print(f"Saved plot: {output_png}")

    if show_plot:
        plt.show()
    else:
        plt.close()


def plot_field_mean_intensity(point_summary_df, output_png=None, show_plot=True):
    """
    Plot field-level mean mean_r +/- SD over time.
    """
    if len(point_summary_df) == 0:
        print("No point summary to plot: field mean intensity.")
        return

    fig, ax = plt.subplots(figsize=(7, 5))

    ax.errorbar(
        point_summary_df["time_hr"],
        point_summary_df["mean_mean_r"],
        yerr=point_summary_df["std_mean_r"],
        marker="o",
        linewidth=2,
        capsize=4
    )

    ax.set_xlabel("Time (hr)")
    ax.set_ylabel("mean mean_r")
    ax.set_title("Field mean mean_r over time")
    ax.grid(alpha=0.2)

    plt.tight_layout()

    if output_png is not None:
        plt.savefig(output_png, dpi=300, bbox_inches="tight")
        print(f"Saved plot: {output_png}")

    if show_plot:
        plt.show()
    else:
        plt.close()


def plot_gradient_software_style(selected_interval_df, output_png=None, show_plot=True):
    """
    Plot intensity gradient in software-like style:
    - X-axis: midpoint time in hours
    - Y-axis: dI/dt (per hour)
    One line per track_id
    """
    if len(selected_interval_df) == 0:
        print("No interval data to plot.")
        return

    fig, ax = plt.subplots(figsize=(7, 5))

    for track_id, sub_df in selected_interval_df.groupby("track_id", dropna=True):
        sub_df = sub_df.sort_values("time_mid_hr")
        ax.plot(
            sub_df["time_mid_hr"],
            sub_df["dI_dt"],
            marker="o",
            linewidth=1,
            alpha=0.8
        )

    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.set_xlabel("Time (hr)")
    ax.set_ylabel("Intensity gradient (dI/dt)")
    ax.set_title("Intensity Gradient (software-style)")
    ax.grid(alpha=0.2)

    plt.tight_layout()

    if output_png is not None:
        plt.savefig(output_png, dpi=300, bbox_inches="tight")
        print(f"Saved plot: {output_png}")

    if show_plot:
        plt.show()
    else:
        plt.close()


def plot_individual_gradient(selected_interval_df, output_png=None, show_plot=True):
    """
    Plot dI/dt vs midpoint time for each selected track.
    Uses interval-level table.
    """
    if len(selected_interval_df) == 0:
        print("No selected interval data to plot: individual gradient.")
        return

    fig, ax = plt.subplots(figsize=(7, 5))

    for track_id, sub_df in selected_interval_df.groupby("track_id", dropna=True):
        sub_df = sub_df.sort_values("time_mid_hr")
        ax.plot(
            sub_df["time_mid_hr"],
            sub_df["dI_dt"],
            marker="o",
            linewidth=1,
            alpha=0.7
        )

    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.set_xlabel("Time (hr)")
    ax.set_ylabel("dI/dt")
    ax.set_title("Individual blob dI/dt over time")
    ax.grid(alpha=0.2)

    plt.tight_layout()

    if output_png is not None:
        plt.savefig(output_png, dpi=300, bbox_inches="tight")
        print(f"Saved plot: {output_png}")

    if show_plot:
        plt.show()
    else:
        plt.close()


def plot_field_mean_gradient(interval_summary_df, output_png=None, show_plot=True):
    """
    Plot field-level mean dI/dt +/- SD over time.
    """
    if len(interval_summary_df) == 0:
        print("No interval summary to plot: field mean gradient.")
        return

    fig, ax = plt.subplots(figsize=(7, 5))

    ax.errorbar(
        interval_summary_df["time_mid_hr"],
        interval_summary_df["mean_dI_dt"],
        yerr=interval_summary_df["std_dI_dt"],
        marker="o",
        linewidth=2,
        capsize=4
    )

    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.set_xlabel("Time (hr)")
    ax.set_ylabel("mean dI/dt")
    ax.set_title("Field mean dI/dt over time")
    ax.grid(alpha=0.2)

    plt.tight_layout()

    if output_png is not None:
        plt.savefig(output_png, dpi=300, bbox_inches="tight")
        print(f"Saved plot: {output_png}")

    if show_plot:
        plt.show()
    else:
        plt.close()

In [ ]:
def plot_blob_kinetics_4panel(
    selected_point_df,
    selected_interval_df,
    point_summary_df,
    interval_summary_df,
    config,
    output_png=None,
    show_plot=True
):
    """
    4-panel summary figure:
    A. Individual mean_r
    B. Field mean mean_r ± SD
    C. Individual dI/dt
    D. Field mean dI/dt ± SD
    """
    if len(selected_point_df) == 0 and len(selected_interval_df) == 0:
        print("Insufficient data for 4-panel plot.")
        return

    intensity_col = config["intensity_col"]

    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    ax1, ax2, ax3, ax4 = axes.flatten()

    # Panel A: individual mean_r
    if len(selected_point_df) > 0:
        for track_id, sub_df in selected_point_df.groupby("track_id", dropna=True):
            sub_df = sub_df.sort_values("time_hr")
            ax1.plot(
                sub_df["time_hr"],
                sub_df[intensity_col],
                marker="o",
                linewidth=1,
                alpha=0.6
            )
    ax1.set_title("A. Individual mean_r")
    ax1.set_xlabel("Time (hr)")
    ax1.set_ylabel(intensity_col)
    ax1.grid(alpha=0.2)

    # Panel B: field mean mean_r
    if len(point_summary_df) > 0:
        ax2.errorbar(
            point_summary_df["time_hr"],
            point_summary_df["mean_mean_r"],
            yerr=point_summary_df["std_mean_r"],
            marker="o",
            linewidth=2,
            capsize=4
        )
    ax2.set_title("B. Field mean mean_r ± SD")
    ax2.set_xlabel("Time (hr)")
    ax2.set_ylabel("mean mean_r")
    ax2.grid(alpha=0.2)

    # Panel C: individual dI/dt
    if len(selected_interval_df) > 0:
        for track_id, sub_df in selected_interval_df.groupby("track_id", dropna=True):
            sub_df = sub_df.sort_values("time_mid_hr")
            ax3.plot(
                sub_df["time_mid_hr"],
                sub_df["dI_dt"],
                marker="o",
                linewidth=1,
                alpha=0.6
            )
    ax3.axhline(0, color="black", linestyle="--", linewidth=1)
    ax3.set_title("C. Individual dI/dt")
    ax3.set_xlabel("Time (hr)")
    ax3.set_ylabel("dI/dt")
    ax3.grid(alpha=0.2)

    # Panel D: field mean dI/dt
    if len(interval_summary_df) > 0:
        ax4.errorbar(
            interval_summary_df["time_mid_hr"],
            interval_summary_df["mean_dI_dt"],
            yerr=interval_summary_df["std_dI_dt"],
            marker="o",
            linewidth=2,
            capsize=4
        )
    ax4.axhline(0, color="black", linestyle="--", linewidth=1)
    ax4.set_title("D. Field mean dI/dt ± SD")
    ax4.set_xlabel("Time (hr)")
    ax4.set_ylabel("mean dI/dt")
    ax4.grid(alpha=0.2)

    plt.tight_layout()

    if output_png is not None:
        plt.savefig(output_png, dpi=300, bbox_inches="tight")
        print(f"Saved plot: {output_png}")

    if show_plot:
        plt.show()
    else:
        plt.close()

In [ ]:
def run_individual_blob_kinetics_one_field(input_file, output_dir, config, stable_eps=1e-6):
    """
    Run the full pipeline for one field using manuscript/software-style kinetics:
    - read input
    - isolation
    - tracking
    - continuous-track selection
    - point-level kinetics
    - interval-level kinetics
    - summaries
    - Excel export
    - plots export
    """
    input_file = Path(input_file)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    base_name = input_file.stem

    print(f"\n=== Processing: {input_file.name} ===")

    # read input
    raw_df = pd.read_excel(input_file)

    # isolation
    iso_df = compute_isolation_metrics_all_frames(raw_df, config)
    iso_df = mark_isolated_blobs(iso_df, config)

    # tracking
    tracked_df = build_blob_tracks(iso_df, config)
    tracked_df, selected_df = select_continuous_tracks(tracked_df, config)

    # kinetics
    selected_point_df, selected_interval_df = compute_individual_kinetics_manuscript_style(
        selected_df, config, stable_eps=stable_eps
    )

    # summaries
    track_summary_df = build_track_level_summary(
        selected_point_df, selected_interval_df, config
    )

    point_summary_df, interval_summary_df = build_timepoint_summary(
        selected_point_df, selected_interval_df, config
    )

    direction_summary_df = build_direction_summary_from_intervals(
        selected_interval_df, stable_eps=stable_eps
    )

    selection_qc_df = build_selection_qc_table(
        raw_df, iso_df, tracked_df, selected_point_df, selected_interval_df
    )

    # export Excel
    excel_path = output_dir / f"{base_name}_individual_blob_kinetics.xlsx"
    export_blob_kinetics_excel(
        raw_df=raw_df,
        iso_df=iso_df,
        tracked_df=tracked_df,
        selected_point_df=selected_point_df,
        selected_interval_df=selected_interval_df,
        track_summary_df=track_summary_df,
        point_summary_df=point_summary_df,
        interval_summary_df=interval_summary_df,
        direction_summary_df=direction_summary_df,
        selection_qc_df=selection_qc_df,
        output_path=excel_path
    )

    # export plots
    if config.get("save_plots", True):
        plot_individual_intensity(
            selected_point_df,
            config,
            output_png=output_dir / f"{base_name}_individual_mean_r.png",
            show_plot=config.get("show_plots", True)
        )

        plot_field_mean_intensity(
            point_summary_df,
            output_png=output_dir / f"{base_name}_field_mean_r.png",
            show_plot=config.get("show_plots", True)
        )

        plot_gradient_software_style(
            selected_interval_df,
            output_png=output_dir / f"{base_name}_gradient_software_style.png",
            show_plot=config.get("show_plots", True)
        )

        plot_individual_gradient(
            selected_interval_df,
            output_png=output_dir / f"{base_name}_individual_dIdt.png",
            show_plot=config.get("show_plots", True)
        )

        plot_field_mean_gradient(
            interval_summary_df,
            output_png=output_dir / f"{base_name}_field_dIdt.png",
            show_plot=config.get("show_plots", True)
        )

        plot_blob_kinetics_4panel(
            selected_point_df,
            selected_interval_df,
            point_summary_df,
            interval_summary_df,
            config,
            output_png=output_dir / f"{base_name}_blob_kinetics_4panel.png",
            show_plot=config.get("show_plots", True)
        )

    result = {
        "input_file": input_file,
        "raw_df": raw_df,
        "iso_df": iso_df,
        "tracked_df": tracked_df,
        "selected_df": selected_df,
        "selected_point_df": selected_point_df,
        "selected_interval_df": selected_interval_df,
        "track_summary_df": track_summary_df,
        "point_summary_df": point_summary_df,
        "interval_summary_df": interval_summary_df,
        "direction_summary_df": direction_summary_df,
        "selection_qc_df": selection_qc_df,
        "excel_path": excel_path,
    }

    return result

In [ ]:
def run_individual_blob_kinetics_batch(input_files, output_dir, config, stable_eps=1e-6):
    """
    Run the full pipeline for multiple input files.
    Returns:
    - summary_df
    - all_results
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    all_results = []
    summary_rows = []

    for input_file in input_files:
        try:
            result = run_individual_blob_kinetics_one_field(
                input_file=input_file,
                output_dir=output_dir,
                config=config,
                stable_eps=stable_eps
            )

            selected_point_df = result["selected_point_df"]
            selected_interval_df = result["selected_interval_df"]

            summary_rows.append({
                "input_file": Path(input_file).name,
                "n_raw_rows": len(result["raw_df"]),
                "n_isolated_rows": int(result["iso_df"]["is_isolated"].sum()) if "is_isolated" in result["iso_df"].columns else np.nan,
                "n_selected_points": len(selected_point_df),
                "n_selected_intervals": len(selected_interval_df),
                "n_selected_tracks": selected_point_df["track_id"].nunique() if len(selected_point_df) > 0 else 0,
                "excel_path": str(result["excel_path"]),
                "status": "ok",
            })

            all_results.append(result)

        except Exception as e:
            summary_rows.append({
                "input_file": Path(input_file).name,
                "n_raw_rows": np.nan,
                "n_isolated_rows": np.nan,
                "n_selected_points": np.nan,
                "n_selected_intervals": np.nan,
                "n_selected_tracks": np.nan,
                "excel_path": "",
                "status": f"error: {e}",
            })

    summary_df = pd.DataFrame(summary_rows)
    return summary_df, all_results

In [ ]:
if len(input_files) > 0:
    summary_df, all_results = run_individual_blob_kinetics_batch(
        input_files=input_files,
        output_dir=OUTPUT_DIR,
        config=CONFIG,
        stable_eps=1e-6
    )

    print("\nBatch summary:")
    display(summary_df)
else:
    print("No input files found yet.")

In [ ]:
if len(input_files) > 0:
    summary_df, all_results = run_individual_blob_kinetics_batch(
        input_files=input_files,
        output_dir=OUTPUT_DIR,
        config=CONFIG
    )

    print("\nBatch summary:")
    display(summary_df)
else:
    print("No input files found yet.")

In [ ]:
if len(input_files) > 0:
    test_file = input_files[0]
    print("Testing Cells 12–14 on:", test_file.name)

    raw_df = pd.read_excel(test_file)

    # upstream pipeline
    iso_df = compute_isolation_metrics_all_frames(raw_df, CONFIG)
    iso_df = mark_isolated_blobs(iso_df, CONFIG)
    tracked_df = build_blob_tracks(iso_df, CONFIG)
    tracked_df, selected_df = select_continuous_tracks(tracked_df, CONFIG)

    # new kinetics
    selected_point_df, selected_interval_df = compute_individual_kinetics_manuscript_style(
        selected_df, CONFIG
    )

    track_summary_df = build_track_level_summary(
        selected_point_df, selected_interval_df, CONFIG
    )

    point_summary_df, interval_summary_df = build_timepoint_summary(
        selected_point_df, selected_interval_df, CONFIG
    )

    print("\nselected_point_df preview:")
    display(selected_point_df.head(10))

    print("\nselected_interval_df preview:")
    display(selected_interval_df.head(10))

    print("\ntrack_summary_df preview:")
    display(track_summary_df.head(10))

    print("\npoint_summary_df preview:")
    display(point_summary_df.head(10))

    print("\ninterval_summary_df preview:")
    display(interval_summary_df.head(10))
else:
    print("No input files found yet.")

In [ ]:
if len(input_files) > 0:
    test_file = input_files[0]
    print("Testing Cells 15–17 on:", test_file.name)

    raw_df = pd.read_excel(test_file)

    iso_df = compute_isolation_metrics_all_frames(raw_df, CONFIG)
    iso_df = mark_isolated_blobs(iso_df, CONFIG)
    tracked_df = build_blob_tracks(iso_df, CONFIG)
    tracked_df, selected_df = select_continuous_tracks(tracked_df, CONFIG)

    selected_point_df, selected_interval_df = compute_individual_kinetics_manuscript_style(
        selected_df, CONFIG
    )

    track_summary_df = build_track_level_summary(
        selected_point_df, selected_interval_df, CONFIG
    )

    point_summary_df, interval_summary_df = build_timepoint_summary(
        selected_point_df, selected_interval_df, CONFIG
    )

    direction_summary_df = build_direction_summary_from_intervals(
        selected_interval_df
    )

    selection_qc_df = build_selection_qc_table(
        raw_df, iso_df, tracked_df, selected_point_df, selected_interval_df
    )

    print("\ndirection_summary_df preview:")
    display(direction_summary_df.head(10))

    print("\nselection_qc_df:")
    display(selection_qc_df)

    # quick plots
    plot_individual_intensity(selected_point_df, CONFIG, show_plot=True)
    plot_gradient_software_style(selected_interval_df, show_plot=True)
    plot_field_mean_intensity(point_summary_df, show_plot=True)
    plot_field_mean_gradient(interval_summary_df, show_plot=True)
else:
    print("No input files found yet.")

In [ ]:
if len(input_files) > 0:
    one_result = run_individual_blob_kinetics_one_field(
        input_file=input_files[0],
        output_dir=OUTPUT_DIR,
        config=CONFIG,
        stable_eps=1e-6
    )

    print("\nExcel saved to:")
    print(one_result["excel_path"])

    print("\nTrack summary preview:")
    display(one_result["track_summary_df"].head())

    print("\nPoint summary preview:")
    display(one_result["point_summary_df"].head())

    print("\nInterval summary preview:")
    display(one_result["interval_summary_df"].head())

    print("\nDirection summary preview:")
    display(one_result["direction_summary_df"].head())
else:
    print("No input files found yet.")

In [2]:
track_id_to_view = 1  # <- เปลี่ยนเลข track ที่ต้องการดูได้ตรงนี้

# ----------------------------
# filter selected point table
# ----------------------------
track_point_view = (
    selected_point_df.loc[selected_point_df["track_id"] == track_id_to_view]
    .sort_values("time_hr")
    .reset_index(drop=True)
)

# ----------------------------
# filter selected interval table
# ----------------------------
track_interval_view = (
    selected_interval_df.loc[selected_interval_df["track_id"] == track_id_to_view]
    .sort_values("time_mid_hr")
    .reset_index(drop=True)
)

print(f"Track ID: {track_id_to_view}")
print(f"Number of point rows: {len(track_point_view)}")
print(f"Number of interval rows: {len(track_interval_view)}")

print("\nPoint-level data:")
display(track_point_view)

print("\nInterval-level data:")
display(track_interval_view)

# ----------------------------
# plotting
# ----------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left panel: intensity-time
ax1 = axes[0]
if len(track_point_view) > 0:
    ax1.plot(
        track_point_view["time_hr"],
        track_point_view[CONFIG["intensity_col"]],
        marker="o",
        linewidth=2
    )
ax1.set_title(f"Track {track_id_to_view}: mean_r over time")
ax1.set_xlabel("Time (hr)")
ax1.set_ylabel(CONFIG["intensity_col"])
ax1.grid(alpha=0.2)

# Right panel: gradient-time
ax2 = axes[1]
if len(track_interval_view) > 0:
    ax2.plot(
        track_interval_view["time_mid_hr"],
        track_interval_view["dI_dt"],
        marker="o",
        linewidth=2
    )
ax2.axhline(0, color="black", linestyle="--", linewidth=1)
ax2.set_title(f"Track {track_id_to_view}: dI/dt over time")
ax2.set_xlabel("Time midpoint (hr)")
ax2.set_ylabel("dI/dt")
ax2.grid(alpha=0.2)

plt.tight_layout()
plt.show()

NameError: name 'selected_point_df' is not defined

In [ ]:
track_id_to_view = 3

track_point_view = (
    selected_point_df.loc[selected_point_df["track_id"] == track_id_to_view, [
        "track_id",
        CONFIG["frame_col"],
        "time_min",
        "time_hr",
        CONFIG["intensity_col"],
    ]]
    .sort_values("time_hr")
    .reset_index(drop=True)
)

track_interval_view = (
    selected_interval_df.loc[selected_interval_df["track_id"] == track_id_to_view, [
        "track_id",
        "frame_t",
        "frame_t1",
        "time_mid_hr",
        "I_t",
        "I_t1",
        "delta_intensity",
        "dI_dt",
        "phase_class",
    ]]
    .sort_values("time_mid_hr")
    .reset_index(drop=True)
)

print(f"Track ID: {track_id_to_view}")
print(f"Number of point rows: {len(track_point_view)}")
print(f"Number of interval rows: {len(track_interval_view)}")

print("\nPoint-level data:")
display(track_point_view)

print("\nInterval-level data:")
display(track_interval_view)

In [ ]:
CONFIG["save_plots"] = False
CONFIG["show_plots"] = False

In [ ]:
import re

# optional: excel only mode
CONFIG["save_plots"] = False
CONFIG["show_plots"] = False

def extract_replicate_name(file_path):
    """
    Extract replicate label from filename stem.
    Example:
    'WT replicate 3 field 1' -> 'WT replicate 3'
    """
    stem = Path(file_path).stem
    m = re.search(r"^(.*?replicate\s+\d+)", stem, flags=re.IGNORECASE)
    if m:
        return m.group(1).strip()
    return None


# group files by replicate
replicate_to_files = {}

for f in input_files:
    rep_name = extract_replicate_name(f)
    if rep_name is not None:
        replicate_to_files.setdefault(rep_name, []).append(f)

# show grouping preview
print("Detected replicate groups:")
for rep_name, files in sorted(replicate_to_files.items()):
    print(f"\n{rep_name}")
    for f in sorted(files, key=lambda x: x.name):
        print(" -", f.name)

# run all replicate groups
all_replicate_summaries = []

for rep_name, files in sorted(replicate_to_files.items()):
    rep_output_dir = OUTPUT_DIR / rep_name
    rep_output_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"Running replicate group: {rep_name}")
    print(f"Output folder: {rep_output_dir}")
    print(f"Number of files: {len(files)}")
    print(f"{'='*60}")

    summary_df, all_results = run_individual_blob_kinetics_batch(
        input_files=sorted(files, key=lambda x: x.name),
        output_dir=rep_output_dir,
        config=CONFIG,
        stable_eps=1e-6
    )

    summary_df = summary_df.copy()
    summary_df["replicate_name"] = rep_name
    all_replicate_summaries.append(summary_df)

# combine all summaries
if len(all_replicate_summaries) > 0:
    grand_summary_df = pd.concat(all_replicate_summaries, ignore_index=True)

    print("\nGrand summary:")
    display(grand_summary_df)
else:
    print("No replicate groups detected.")

In [ ]:
print("Input files:")
for f in input_files:
    print("-", f.name)

In [ ]:
from pathlib import Path

INPUT_DIR = Path("inputs")  # เปลี่ยนถ้าโฟลเดอร์ input ของคุณชื่อไม่ใช่ inputs
input_files = sorted(INPUT_DIR.glob("*.xlsx"))

print("Input files:")
for f in input_files:
    print("-", f.name)

print(f"\nTotal files: {len(input_files)}")

In [ ]:
print(INPUT_DIR.resolve())

In [ ]:
from pathlib import Path

INPUT_DIR = Path("inputs") / "individual_blob_kinetics"
input_files = sorted(INPUT_DIR.glob("*.xlsx"))

print("INPUT_DIR:", INPUT_DIR.resolve())
print("\nInput files:")
for f in input_files:
    print("-", f.name)

print(f"\nTotal files: {len(input_files)}")

In [4]:
from pathlib import Path
import shutil
import pandas as pd
import matplotlib.pyplot as plt

from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage


# ============================================================
# User settings
# ============================================================
BASE_OUTPUT_DIR = Path("outputs") / "individual_blob_kinetics"

# Set to None to process all replicate folders under BASE_OUTPUT_DIR
REPLICATE_FILTER = "WT replicate 3"

# If True, skip files that already have output_with_track_plots file
SKIP_IF_OUTPUT_EXISTS = True


# ============================================================
# Discover files
# ============================================================
excel_files = sorted(BASE_OUTPUT_DIR.rglob("*_individual_blob_kinetics.xlsx"))

# Exclude already generated track-plot outputs
excel_files = [
    f for f in excel_files
    if "output_with_track_plots" not in f.name
]

# Keep only files inside the selected replicate folder
if REPLICATE_FILTER is not None:
    excel_files = [
        f for f in excel_files
        if f.parent.name.lower() == REPLICATE_FILTER.lower()
    ]

print("Selected Excel files:")
for f in excel_files:
    print("-", f)

print(f"\nTotal selected files: {len(excel_files)}")


# ============================================================
# Helper: make one track plot
# ============================================================
def make_track_plot(track_df, track_id, save_path):
    track_df = track_df.sort_values("time_mid_hr").copy()

    x = track_df["time_mid_hr"].values
    y = track_df["dI_dt"].values

    colors = ["forestgreen" if val > 0 else "firebrick" for val in y]

    fig, ax = plt.subplots(figsize=(8, 3.6))

    if len(x) >= 2:
        diffs = pd.Series(x).diff().dropna()
        min_step = diffs.min() if len(diffs) > 0 else 0.5
        bar_width = max(min_step * 0.7, 0.1)
    else:
        bar_width = 0.3

    ax.bar(
        x, y,
        width=bar_width,
        color=colors,
        edgecolor="black",
        linewidth=0.6
    )

    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.set_title(f"track ID {track_id}", fontsize=12, pad=10)
    ax.set_xlabel("Time midpoint (hr)", fontsize=10)
    ax.set_ylabel("dI/dt", fontsize=10)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", alpha=0.2, linestyle=":")

    plt.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


# ============================================================
# Helper: process one Excel file
# ============================================================
def insert_track_plots_into_excel(input_excel_path):
    input_excel_path = Path(input_excel_path)
    output_excel_path = input_excel_path.with_name(
        input_excel_path.stem + "_output_with_track_plots.xlsx"
    )

    if SKIP_IF_OUTPUT_EXISTS and output_excel_path.exists():
        return {
            "input_excel": str(input_excel_path),
            "output_excel": str(output_excel_path),
            "n_interval_rows": None,
            "n_tracks_total": None,
            "plots_inserted": None,
            "tracks_skipped": None,
            "status": "skipped_existing_output",
        }

    # ----------------------------
    # read interval sheet
    # ----------------------------
    sheet_name = "selected_blob_intervals"
    df = pd.read_excel(input_excel_path, sheet_name=sheet_name)

    if "track_ID" in df.columns:
        track_col = "track_ID"
    elif "track_id" in df.columns:
        track_col = "track_id"
    else:
        raise ValueError(
            f"{input_excel_path.name}: missing track column "
            f"(expected 'track_ID' or 'track_id')"
        )

    required_cols = [track_col, "time_mid_hr", "dI_dt", "phase_class"]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(
            f"{input_excel_path.name}: missing required columns {missing_cols}"
        )

    df = df.dropna(subset=[track_col, "time_mid_hr", "dI_dt"]).copy()
    df = df.sort_values([track_col, "time_mid_hr"]).reset_index(drop=True)

    # ----------------------------
    # open workbook
    # ----------------------------
    wb = load_workbook(input_excel_path)

    if "track_plots" in wb.sheetnames:
        del wb["track_plots"]

    ws = wb.create_sheet("track_plots")
    ws["A1"] = "Single-particle intensity-gradient plots"
    ws["A2"] = f"Source file: {input_excel_path.name}"
    ws["A3"] = "Sheet used: selected_blob_intervals"

    # ----------------------------
    # create cache folder for PNGs
    # ----------------------------
    cache_dir = input_excel_path.parent / f"_{input_excel_path.stem}_track_plot_cache"
    if cache_dir.exists():
        shutil.rmtree(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)

    insert_start_row = 5
    row_spacing = 22
    plots_inserted = 0
    tracks_skipped = 0

    try:
        for track_id, track_df in df.groupby(track_col, dropna=True):
            track_df = track_df.sort_values("time_mid_hr").copy()

            if len(track_df) < 2:
                tracks_skipped += 1
                continue

            png_path = cache_dir / f"track_{track_id}.png"
            make_track_plot(track_df, track_id, png_path)

            if not png_path.exists():
                raise FileNotFoundError(f"Plot PNG was not created: {png_path}")

            img = XLImage(str(png_path))
            img.width = 720
            img.height = 300

            anchor_cell = f"A{insert_start_row + plots_inserted * row_spacing}"
            ws.add_image(img, anchor_cell)
            plots_inserted += 1

        wb.save(output_excel_path)

    finally:
        # Clean up cache after workbook is saved
        if cache_dir.exists():
            shutil.rmtree(cache_dir, ignore_errors=True)

    return {
        "input_excel": str(input_excel_path),
        "output_excel": str(output_excel_path),
        "n_interval_rows": len(df),
        "n_tracks_total": df[track_col].nunique(),
        "plots_inserted": plots_inserted,
        "tracks_skipped": tracks_skipped,
        "status": "ok",
    }


# ============================================================
# Run batch with progress
# ============================================================
batch_results = []
n_total = len(excel_files)

for i, f in enumerate(excel_files, start=1):
    print(f"\n[{i}/{n_total}] Processing: {f.name}")

    try:
        result = insert_track_plots_into_excel(f)
        batch_results.append(result)

        print(f"    status         : {result['status']}")
        print(f"    output_excel   : {result['output_excel']}")

        if result["status"] == "ok":
            print(f"    interval_rows  : {result['n_interval_rows']}")
            print(f"    total_tracks   : {result['n_tracks_total']}")
            print(f"    plots_inserted : {result['plots_inserted']}")
            print(f"    tracks_skipped : {result['tracks_skipped']}")

    except Exception as e:
        batch_results.append({
            "input_excel": str(f),
            "output_excel": "",
            "n_interval_rows": None,
            "n_tracks_total": None,
            "plots_inserted": None,
            "tracks_skipped": None,
            "status": f"error: {e}",
        })
        print(f"    status         : error")
        print(f"    message        : {e}")

batch_results_df = pd.DataFrame(batch_results)

print("\nBatch results summary:")
display(batch_results_df)

Selected Excel files:
- outputs\individual_blob_kinetics\WT replicate 3\WT replicate 3 field 1_individual_blob_kinetics.xlsx
- outputs\individual_blob_kinetics\WT replicate 3\WT replicate 3 field 2_individual_blob_kinetics.xlsx
- outputs\individual_blob_kinetics\WT replicate 3\WT replicate 3 field 3_individual_blob_kinetics.xlsx

Total selected files: 3

[1/3] Processing: WT replicate 3 field 1_individual_blob_kinetics.xlsx
    status         : skipped_existing_output
    output_excel   : outputs\individual_blob_kinetics\WT replicate 3\WT replicate 3 field 1_individual_blob_kinetics_output_with_track_plots.xlsx

[2/3] Processing: WT replicate 3 field 2_individual_blob_kinetics.xlsx
    status         : skipped_existing_output
    output_excel   : outputs\individual_blob_kinetics\WT replicate 3\WT replicate 3 field 2_individual_blob_kinetics_output_with_track_plots.xlsx

[3/3] Processing: WT replicate 3 field 3_individual_blob_kinetics.xlsx
    status         : skipped_existing_output
 

,input_excel,output_excel,n_interval_rows,n_tracks_total,plots_inserted,tracks_skipped,status
0,outputs\individual_blob_kinetics\WT replicate ...,outputs\individual_blob_kinetics\WT replicate ...,None,None,None,None,skipped_existing_output
1,outputs\individual_blob_kinetics\WT replicate ...,outputs\individual_blob_kinetics\WT replicate ...,None,None,None,None,skipped_existing_output
2,outputs\individual_blob_kinetics\WT replicate ...,outputs\individual_blob_kinetics\WT replicate ...,None,None,None,None,skipped_existing_output


In [15]:
from pathlib import Path
import shutil
import pandas as pd
import matplotlib.pyplot as plt

from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage


# ============================================================
# User settings
# ============================================================
BASE_OUTPUT_DIR = Path("outputs") / "individual_blob_kinetics"

# Example:
# REPLICATE_FILTER = "WT replicate 3"
# Set to None to process all replicate folders
REPLICATE_FILTER = "WT replicate 3"

# If True, skip output file if it already exists
SKIP_IF_OUTPUT_EXISTS = True

# Test one file first before full batch
TEST_MODE = True

# If TEST_MODE = True, set one file here
TEST_FILE = Path(
    r"outputs\individual_blob_kinetics\WT replicate 3\WT replicate 3 field 1_individual_blob_kinetics.xlsx"
)

# Keep cache images for debugging
KEEP_CACHE = True


# ============================================================
# Helper: one track plot
# ============================================================
def make_track_plot(track_df, track_id, save_path):
    """
    Create a manuscript-style bar plot for one track.
    X-axis = time_mid_hr
    Y-axis = dI_dt
    Green = positive dI_dt
    Red = negative dI_dt

    Improvements:
    - narrower bars centered at time_mid_hr
    - x ticks aligned to actual time points
    - data labels on each bar
    - fixed y-axis range for all plots
    """
    track_df = track_df.sort_values("time_mid_hr").copy()

    x = track_df["time_mid_hr"].astype(float).values
    y = track_df["dI_dt"].astype(float).values

    colors = ["forestgreen" if val > 0 else "firebrick" for val in y]

    fig, ax = plt.subplots(figsize=(8.5, 3.8))

    # make bars narrower so they visually sit on the correct time position
    if len(x) >= 2:
        diffs = pd.Series(x).diff().dropna()
        min_step = diffs.min() if len(diffs) > 0 else 0.5
        bar_width = max(min_step * 0.55, 0.09)
    else:
        bar_width = 0.08

    bars = ax.bar(
        x,
        y,
        width=bar_width,
        color=colors,
        edgecolor="black",
        linewidth=0.6,
        align="center"
    )

    # zero line
    ax.axhline(0, color="black", linestyle="--", linewidth=1)

    # fixed y-axis for all plots
    ax.set_ylim(-0.25, 0.25)

    # title and labels
    ax.set_title(f"track ID {track_id}", fontsize=12, pad=10)
    ax.set_xlabel("Time midpoint (hr)", fontsize=10)
    ax.set_ylabel("dI/dt", fontsize=10)

    # manuscript-like styling
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", alpha=0.2, linestyle=":")

    # set x ticks exactly at observed time points
    ax.set_xticks(x)
    ax.set_xticklabels([f"{v:.2f}" for v in x], rotation=45, ha="right")

    # data labels on each bar
    offset = 0.015  # fixed offset works better with fixed y-limits

    for xi, yi in zip(x, y):
        if yi >= 0:
            va = "bottom"
            y_text = min(yi + offset, 0.245)
        else:
            va = "top"
            y_text = max(yi - offset, -0.245)

        ax.text(
            xi,
            y_text,
            f"{yi:.4f}",
            ha="center",
            va=va,
            fontsize=8
        )

    plt.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

In [6]:
def insert_track_plots_into_excel(input_excel_path, keep_cache=True, skip_if_output_exists=True):
    """
    Read selected_blob_intervals from one Excel file,
    generate one plot per track, insert into a new worksheet 'track_plots',
    and save as a new Excel file.

    Output file:
        <original_stem>_output_with_track_plots.xlsx
    """
    input_excel_path = Path(input_excel_path)

    if not input_excel_path.exists():
        raise FileNotFoundError(f"Input Excel file not found: {input_excel_path}")

    output_excel_path = input_excel_path.with_name(
        input_excel_path.stem + "_output_with_track_plots.xlsx"
    )

    if skip_if_output_exists and output_excel_path.exists():
        return {
            "input_excel": str(input_excel_path),
            "output_excel": str(output_excel_path),
            "n_interval_rows": None,
            "n_tracks_total": None,
            "plots_inserted": None,
            "tracks_skipped": None,
            "status": "skipped_existing_output",
            "cache_dir": "",
        }

    # ----------------------------
    # read interval sheet
    # ----------------------------
    sheet_name = "selected_blob_intervals"
    df = pd.read_excel(input_excel_path, sheet_name=sheet_name)

    # support both track_ID and track_id
    if "track_ID" in df.columns:
        track_col = "track_ID"
    elif "track_id" in df.columns:
        track_col = "track_id"
    else:
        raise ValueError(
            f"{input_excel_path.name}: missing track column "
            f"(expected 'track_ID' or 'track_id')"
        )

    required_cols = [track_col, "time_mid_hr", "dI_dt", "phase_class"]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(
            f"{input_excel_path.name}: missing required columns {missing_cols}"
        )

    df = df.dropna(subset=[track_col, "time_mid_hr", "dI_dt"]).copy()
    df = df.sort_values([track_col, "time_mid_hr"]).reset_index(drop=True)

    # ----------------------------
    # open workbook
    # ----------------------------
    wb = load_workbook(input_excel_path)

    if "track_plots" in wb.sheetnames:
        del wb["track_plots"]

    ws = wb.create_sheet("track_plots")
    ws["A1"] = "Single-particle intensity-gradient plots"
    ws["A2"] = f"Source file: {input_excel_path.name}"
    ws["A3"] = "Sheet used: selected_blob_intervals"

    # ----------------------------
    # create cache folder for PNGs
    # ----------------------------
    cache_dir = input_excel_path.parent / f"_{input_excel_path.stem}_track_plot_cache"

    if cache_dir.exists():
        shutil.rmtree(cache_dir, ignore_errors=True)
    cache_dir.mkdir(parents=True, exist_ok=True)

    insert_start_row = 5
    row_spacing = 22
    plots_inserted = 0
    tracks_skipped = 0

    # keep image refs alive until save
    image_refs = []

    try:
        for track_id, track_df in df.groupby(track_col, dropna=True):
            track_df = track_df.sort_values("time_mid_hr").copy()

            # skip short tracks
            if len(track_df) < 2:
                tracks_skipped += 1
                continue

            png_path = cache_dir / f"track_{track_id}.png"
            make_track_plot(track_df, track_id, png_path)

            if not png_path.exists():
                raise FileNotFoundError(f"Plot PNG was not created: {png_path}")

            img = XLImage(str(png_path))
            img.width = 720
            img.height = 300

            anchor_cell = f"A{insert_start_row + plots_inserted * row_spacing}"
            ws.add_image(img, anchor_cell)
            image_refs.append(img)

            plots_inserted += 1

        wb.save(output_excel_path)

    finally:
        if keep_cache:
            print(f"Track plot cache kept at: {cache_dir}")
        else:
            if cache_dir.exists():
                shutil.rmtree(cache_dir, ignore_errors=True)

    return {
        "input_excel": str(input_excel_path),
        "output_excel": str(output_excel_path),
        "n_interval_rows": len(df),
        "n_tracks_total": df[track_col].nunique(),
        "plots_inserted": plots_inserted,
        "tracks_skipped": tracks_skipped,
        "status": "ok",
        "cache_dir": str(cache_dir),
    }

In [16]:
# ============================================================
# Discover files
# ============================================================
excel_files = sorted(BASE_OUTPUT_DIR.rglob("*_individual_blob_kinetics.xlsx"))

# exclude already generated files
excel_files = [
    f for f in excel_files
    if "output_with_track_plots" not in f.name
]

# keep only files inside the exact replicate folder
if REPLICATE_FILTER is not None:
    excel_files = [
        f for f in excel_files
        if f.parent.name.lower() == REPLICATE_FILTER.lower()
    ]

print("Selected Excel files:")
for f in excel_files:
    print("-", f)
print(f"\nTotal selected files: {len(excel_files)}")


# ============================================================
# Run test or batch
# ============================================================
if TEST_MODE:
    print("\nTEST MODE: processing one file only")
    print("Test file:", TEST_FILE)

    result = insert_track_plots_into_excel(
        TEST_FILE,
        keep_cache=KEEP_CACHE,
        skip_if_output_exists=False
    )

    print("\nTest result:")
    print(result)

else:
    batch_results = []
    n_total = len(excel_files)

    for i, f in enumerate(excel_files, start=1):
        print(f"\n[{i}/{n_total}] Processing: {f.name}")

        try:
            result = insert_track_plots_into_excel(
                f,
                keep_cache=KEEP_CACHE,
                skip_if_output_exists=SKIP_IF_OUTPUT_EXISTS
            )
            batch_results.append(result)

            print(f"    status         : {result['status']}")
            print(f"    output_excel   : {result['output_excel']}")

            if result["status"] == "ok":
                print(f"    interval_rows  : {result['n_interval_rows']}")
                print(f"    total_tracks   : {result['n_tracks_total']}")
                print(f"    plots_inserted : {result['plots_inserted']}")
                print(f"    tracks_skipped : {result['tracks_skipped']}")

        except Exception as e:
            batch_results.append({
                "input_excel": str(f),
                "output_excel": "",
                "n_interval_rows": None,
                "n_tracks_total": None,
                "plots_inserted": None,
                "tracks_skipped": None,
                "status": f"error: {e}",
                "cache_dir": "",
            })
            print(f"    status         : error")
            print(f"    message        : {e}")

    batch_results_df = pd.DataFrame(batch_results)

    print("\nBatch results summary:")
    display(batch_results_df)

Selected Excel files:
- outputs\individual_blob_kinetics\WT replicate 3\WT replicate 3 field 1_individual_blob_kinetics.xlsx
- outputs\individual_blob_kinetics\WT replicate 3\WT replicate 3 field 2_individual_blob_kinetics.xlsx
- outputs\individual_blob_kinetics\WT replicate 3\WT replicate 3 field 3_individual_blob_kinetics.xlsx

Total selected files: 3

TEST MODE: processing one file only
Test file: outputs\individual_blob_kinetics\WT replicate 3\WT replicate 3 field 1_individual_blob_kinetics.xlsx
Track plot cache kept at: outputs\individual_blob_kinetics\WT replicate 3\_WT replicate 3 field 1_individual_blob_kinetics_track_plot_cache

Test result:
{'input_excel': 'outputs\\individual_blob_kinetics\\WT replicate 3\\WT replicate 3 field 1_individual_blob_kinetics.xlsx', 'output_excel': 'outputs\\individual_blob_kinetics\\WT replicate 3\\WT replicate 3 field 1_individual_blob_kinetics_output_with_track_plots.xlsx', 'n_interval_rows': 58, 'n_tracks_total': 22, 'plots_inserted': 14, 'tra

In [17]:
from pathlib import Path
import pandas as pd

from openpyxl import load_workbook
from openpyxl.chart import BarChart, Reference
from openpyxl.chart.label import DataLabelList
from openpyxl.styles import Font


def insert_excel_track_charts(input_excel_path, overwrite_output=True):
    """
    Create Excel-native bar charts from 'selected_blob_intervals'
    and insert them into a new worksheet named 'track_plots_excel_chart'.

    Output:
        <original_stem>_with_excel_charts.xlsx
    """
    input_excel_path = Path(input_excel_path)
    output_excel_path = input_excel_path.with_name(
        input_excel_path.stem + "_with_excel_charts.xlsx"
    )

    if output_excel_path.exists() and not overwrite_output:
        return {
            "input_excel": str(input_excel_path),
            "output_excel": str(output_excel_path),
            "n_tracks_total": None,
            "charts_inserted": None,
            "tracks_skipped": None,
            "status": "skipped_existing_output",
        }

    # --------------------------------------------------------
    # Read interval sheet
    # --------------------------------------------------------
    sheet_name = "selected_blob_intervals"
    df = pd.read_excel(input_excel_path, sheet_name=sheet_name)

    if "track_ID" in df.columns:
        track_col = "track_ID"
    elif "track_id" in df.columns:
        track_col = "track_id"
    else:
        raise ValueError(
            f"{input_excel_path.name}: missing track column "
            f"(expected 'track_ID' or 'track_id')"
        )

    required_cols = [track_col, "time_mid_hr", "dI_dt"]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(
            f"{input_excel_path.name}: missing required columns {missing_cols}"
        )

    df = df.dropna(subset=[track_col, "time_mid_hr", "dI_dt"]).copy()
    df = df.sort_values([track_col, "time_mid_hr"]).reset_index(drop=True)

    # --------------------------------------------------------
    # Open workbook
    # --------------------------------------------------------
    wb = load_workbook(input_excel_path)

    sheet_out = "track_plots_excel_chart"
    if sheet_out in wb.sheetnames:
        del wb[sheet_out]

    ws = wb.create_sheet(sheet_out)

    ws["A1"] = "Single-particle intensity-gradient Excel charts"
    ws["A2"] = f"Source file: {input_excel_path.name}"
    ws["A3"] = "Sheet used: selected_blob_intervals"

    ws["A1"].font = Font(bold=True, size=12)
    ws["A2"].font = Font(italic=True)
    ws["A3"].font = Font(italic=True)

    # --------------------------------------------------------
    # Write track-wise data blocks + charts
    # --------------------------------------------------------
    current_row = 5
    charts_inserted = 0
    tracks_skipped = 0

    grouped = df.groupby(track_col, dropna=True)

    for track_id, track_df in grouped:
        track_df = track_df.sort_values("time_mid_hr").copy()

        # skip short tracks
        if len(track_df) < 2:
            tracks_skipped += 1
            continue

        # split positive/negative for Excel coloring
        track_df["dI_dt_pos"] = track_df["dI_dt"].apply(lambda v: v if v > 0 else 0)
        track_df["dI_dt_neg"] = track_df["dI_dt"].apply(lambda v: v if v < 0 else 0)

        # title line for this track
        ws.cell(row=current_row, column=1, value=f"track ID {track_id}")
        ws.cell(row=current_row, column=1).font = Font(bold=True)

        # header row
        header_row = current_row + 1
        ws.cell(row=header_row, column=1, value="time_mid_hr")
        ws.cell(row=header_row, column=2, value="dI_dt_pos")
        ws.cell(row=header_row, column=3, value="dI_dt_neg")
        ws.cell(row=header_row, column=4, value="dI_dt_raw")

        # data rows
        data_start_row = header_row + 1
        for i, (_, row) in enumerate(track_df.iterrows(), start=0):
            r = data_start_row + i
            ws.cell(r, 1, float(row["time_mid_hr"]))
            ws.cell(r, 2, float(row["dI_dt_pos"]))
            ws.cell(r, 3, float(row["dI_dt_neg"]))
            ws.cell(r, 4, float(row["dI_dt"]))

        data_end_row = data_start_row + len(track_df) - 1

        # create chart
        chart = BarChart()
        chart.type = "col"
        chart.style = 2
        chart.title = f"track ID {track_id}"
        chart.y_axis.title = "dI/dt"
        chart.x_axis.title = "time_mid_hr"
        chart.height = 7
        chart.width = 14
        chart.grouping = "clustered"
        chart.overlap = 0
        chart.varyColors = False

        # fixed y-axis range
        chart.y_axis.scaling.min = -0.25
        chart.y_axis.scaling.max = 0.25

        data = Reference(ws, min_col=2, max_col=3, min_row=header_row, max_row=data_end_row)
        cats = Reference(ws, min_col=1, min_row=data_start_row, max_row=data_end_row)

        chart.add_data(data, titles_from_data=True)
        chart.set_categories(cats)

        # add data labels
        chart.dLbls = DataLabelList()
        chart.dLbls.showVal = True

        # color positive series green, negative series red
        if len(chart.series) >= 1:
            chart.series[0].graphicalProperties.solidFill = "228B22"  # forest green
            chart.series[0].graphicalProperties.line.solidFill = "000000"

        if len(chart.series) >= 2:
            chart.series[1].graphicalProperties.solidFill = "B22222"  # firebrick
            chart.series[1].graphicalProperties.line.solidFill = "000000"

        # place chart to the right of the table block
        chart_anchor = f"F{current_row}"
        ws.add_chart(chart, chart_anchor)

        charts_inserted += 1

        # move to next block
        current_row = data_end_row + 18

    # optional column widths
    ws.column_dimensions["A"].width = 14
    ws.column_dimensions["B"].width = 12
    ws.column_dimensions["C"].width = 12
    ws.column_dimensions["D"].width = 12
    ws.column_dimensions["F"].width = 2

    wb.save(output_excel_path)

    return {
        "input_excel": str(input_excel_path),
        "output_excel": str(output_excel_path),
        "n_tracks_total": df[track_col].nunique(),
        "charts_inserted": charts_inserted,
        "tracks_skipped": tracks_skipped,
        "status": "ok",
    }

In [19]:
test_file = Path(r"outputs\individual_blob_kinetics\WT replicate 3\WT replicate 3 field 1_individual_blob_kinetics.xlsx")

result = insert_excel_track_charts(test_file, overwrite_output=True)
print(result)

{'input_excel': 'outputs\\individual_blob_kinetics\\WT replicate 3\\WT replicate 3 field 1_individual_blob_kinetics.xlsx', 'output_excel': 'outputs\\individual_blob_kinetics\\WT replicate 3\\WT replicate 3 field 1_individual_blob_kinetics_with_excel_charts.xlsx', 'n_tracks_total': 22, 'charts_inserted': 14, 'tracks_skipped': 8, 'status': 'ok'}


In [20]:
from pathlib import Path
import pandas as pd

from openpyxl import load_workbook
from openpyxl.chart import BarChart, Reference
from openpyxl.styles import Font


def insert_excel_track_charts_stable(input_excel_path, overwrite_output=True):
    """
    Create simplified and stable Excel-native bar charts from
    'selected_blob_intervals'.

    Key design choices for stability:
    - one raw series only (dI_dt)
    - no data labels
    - no complex series formatting
    - separate data sheet + chart sheet
    - fixed y-axis range across all charts

    Output:
        <original_stem>_with_excel_charts_stable.xlsx
    """
    input_excel_path = Path(input_excel_path)
    output_excel_path = input_excel_path.with_name(
        input_excel_path.stem + "_with_excel_charts_stable.xlsx"
    )

    if output_excel_path.exists() and not overwrite_output:
        return {
            "input_excel": str(input_excel_path),
            "output_excel": str(output_excel_path),
            "n_tracks_total": None,
            "charts_inserted": None,
            "tracks_skipped": None,
            "status": "skipped_existing_output",
        }

    # --------------------------------------------------------
    # Read source data
    # --------------------------------------------------------
    source_sheet = "selected_blob_intervals"
    df = pd.read_excel(input_excel_path, sheet_name=source_sheet)

    if "track_ID" in df.columns:
        track_col = "track_ID"
    elif "track_id" in df.columns:
        track_col = "track_id"
    else:
        raise ValueError(
            f"{input_excel_path.name}: missing track column "
            f"(expected 'track_ID' or 'track_id')"
        )

    required_cols = [track_col, "time_mid_hr", "dI_dt"]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(
            f"{input_excel_path.name}: missing required columns {missing_cols}"
        )

    df = df.dropna(subset=[track_col, "time_mid_hr", "dI_dt"]).copy()
    df = df.sort_values([track_col, "time_mid_hr"]).reset_index(drop=True)

    # --------------------------------------------------------
    # Open workbook
    # --------------------------------------------------------
    wb = load_workbook(input_excel_path)

    for sheet_name in ["track_chart_data_stable", "track_plots_excel_chart_stable"]:
        if sheet_name in wb.sheetnames:
            del wb[sheet_name]

    ws_data = wb.create_sheet("track_chart_data_stable")
    ws_chart = wb.create_sheet("track_plots_excel_chart_stable")

    # --------------------------------------------------------
    # Header for chart sheet
    # --------------------------------------------------------
    ws_chart["A1"] = "Single-particle intensity-gradient Excel charts"
    ws_chart["A2"] = f"Source file: {input_excel_path.name}"
    ws_chart["A3"] = "Chart type: simplified stable Excel-native bar chart"
    ws_chart["A4"] = "Data source: selected_blob_intervals"

    ws_chart["A1"].font = Font(bold=True, size=13)
    ws_chart["A2"].font = Font(italic=True)
    ws_chart["A3"].font = Font(italic=True)
    ws_chart["A4"].font = Font(italic=True)

    ws_chart.sheet_view.showGridLines = False

    # --------------------------------------------------------
    # Populate data sheet and create charts
    # --------------------------------------------------------
    data_row = 1
    chart_row = 6
    charts_inserted = 0
    tracks_skipped = 0

    grouped = df.groupby(track_col, dropna=True)

    for track_id, track_df in grouped:
        track_df = track_df.sort_values("time_mid_hr").copy()

        # keep same rule as original prompt
        if len(track_df) < 2:
            tracks_skipped += 1
            continue

        # ----------------------------
        # Write one data block
        # ----------------------------
        ws_data.cell(row=data_row, column=1, value=f"track ID {track_id}")
        ws_data.cell(row=data_row, column=1).font = Font(bold=True)

        header_row = data_row + 1
        ws_data.cell(row=header_row, column=1, value="time_mid_hr")
        ws_data.cell(row=header_row, column=2, value="dI_dt")

        data_start_row = header_row + 1

        for i, (_, row) in enumerate(track_df.iterrows(), start=0):
            r = data_start_row + i
            ws_data.cell(r, 1, float(row["time_mid_hr"]))
            ws_data.cell(r, 2, float(row["dI_dt"]))

        data_end_row = data_start_row + len(track_df) - 1

        # ----------------------------
        # Create stable chart
        # ----------------------------
        chart = BarChart()
        chart.type = "col"
        chart.style = 2
        chart.title = f"track ID {track_id}"
        chart.y_axis.title = "dI/dt"
        chart.x_axis.title = "Time (hr)"
        chart.height = 9
        chart.width = 18
        chart.grouping = "clustered"
        chart.overlap = 0
        chart.varyColors = False

        # cleaner appearance
        chart.legend = None

        # fixed y-axis range across all charts
        chart.y_axis.scaling.min = -0.25
        chart.y_axis.scaling.max = 0.25

        data_ref = Reference(
            ws_data,
            min_col=2,
            max_col=2,
            min_row=header_row,
            max_row=data_end_row
        )
        cats_ref = Reference(
            ws_data,
            min_col=1,
            min_row=data_start_row,
            max_row=data_end_row
        )

        chart.add_data(data_ref, titles_from_data=True)
        chart.set_categories(cats_ref)

        # Optional simple bar color
        if len(chart.series) >= 1:
            chart.series[0].graphicalProperties.solidFill = "4F81BD"
            chart.series[0].graphicalProperties.line.solidFill = "000000"

        # Add chart to clean chart sheet
        anchor = f"A{chart_row}"
        ws_chart.add_chart(chart, anchor)

        charts_inserted += 1

        # Move to next blocks
        data_row = data_end_row + 3
        chart_row += 18

    # --------------------------------------------------------
    # Format sheets
    # --------------------------------------------------------
    ws_data.column_dimensions["A"].width = 14
    ws_data.column_dimensions["B"].width = 12

    # Keep data sheet visible for transparency/debugging,
    # but you can hide it by uncommenting the next line:
    # ws_data.sheet_state = "hidden"

    wb.save(output_excel_path)

    return {
        "input_excel": str(input_excel_path),
        "output_excel": str(output_excel_path),
        "n_tracks_total": df[track_col].nunique(),
        "charts_inserted": charts_inserted,
        "tracks_skipped": tracks_skipped,
        "status": "ok",
    }

In [21]:
test_file = Path(r"outputs\individual_blob_kinetics\WT replicate 3\WT replicate 3 field 1_individual_blob_kinetics.xlsx")

result = insert_excel_track_charts_stable(test_file, overwrite_output=True)
print(result)

{'input_excel': 'outputs\\individual_blob_kinetics\\WT replicate 3\\WT replicate 3 field 1_individual_blob_kinetics.xlsx', 'output_excel': 'outputs\\individual_blob_kinetics\\WT replicate 3\\WT replicate 3 field 1_individual_blob_kinetics_with_excel_charts_stable.xlsx', 'n_tracks_total': 22, 'charts_inserted': 14, 'tracks_skipped': 8, 'status': 'ok'}


In [22]:
from pathlib import Path
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment


def add_track_manual_plot_layout_sheet(
    input_excel_path,
    output_excel_path=None,
    overwrite_output=True
):
    """
    Add a new worksheet named 'track_manual_plot_layout' to an existing
    individual_blob_kinetics Excel file.

    For each track_id, create two sections:
      1) point data    -> time_min | mean_r
      2) interval data -> time_mid_hr | dI_dt

    This layout is intended to make manual plotting easier.

    Parameters
    ----------
    input_excel_path : str or Path
        Path to existing kinetics Excel output.
    output_excel_path : str or Path or None
        If None, save as:
            <original_stem>_with_track_layout.xlsx
    overwrite_output : bool
        If False and output exists, skip.

    Returns
    -------
    dict
    """
    input_excel_path = Path(input_excel_path)

    if output_excel_path is None:
        output_excel_path = input_excel_path.with_name(
            input_excel_path.stem + "_with_track_layout.xlsx"
        )
    else:
        output_excel_path = Path(output_excel_path)

    if output_excel_path.exists() and not overwrite_output:
        return {
            "input_excel": str(input_excel_path),
            "output_excel": str(output_excel_path),
            "n_tracks_points": None,
            "n_tracks_intervals": None,
            "status": "skipped_existing_output",
        }

    # --------------------------------------------------------
    # Read source sheets
    # --------------------------------------------------------
    points_sheet = "selected_blob_points"
    intervals_sheet = "selected_blob_intervals"

    df_points = pd.read_excel(input_excel_path, sheet_name=points_sheet)
    df_intervals = pd.read_excel(input_excel_path, sheet_name=intervals_sheet)

    # detect track column
    if "track_ID" in df_points.columns:
        track_col_points = "track_ID"
    elif "track_id" in df_points.columns:
        track_col_points = "track_id"
    else:
        raise ValueError(
            f"{input_excel_path.name}: could not find track column in {points_sheet}"
        )

    if "track_ID" in df_intervals.columns:
        track_col_intervals = "track_ID"
    elif "track_id" in df_intervals.columns:
        track_col_intervals = "track_id"
    else:
        raise ValueError(
            f"{input_excel_path.name}: could not find track column in {intervals_sheet}"
        )

    # required columns
    required_points = [track_col_points, "time_min", "mean_r"]
    required_intervals = [track_col_intervals, "time_mid_hr", "dI_dt"]

    missing_points = [c for c in required_points if c not in df_points.columns]
    missing_intervals = [c for c in required_intervals if c not in df_intervals.columns]

    if missing_points:
        raise ValueError(
            f"{input_excel_path.name}: missing columns in {points_sheet}: {missing_points}"
        )

    if missing_intervals:
        raise ValueError(
            f"{input_excel_path.name}: missing columns in {intervals_sheet}: {missing_intervals}"
        )

    # clean/sort
    df_points = df_points.dropna(subset=[track_col_points, "time_min", "mean_r"]).copy()
    df_points = df_points.sort_values([track_col_points, "time_min"]).reset_index(drop=True)

    df_intervals = df_intervals.dropna(subset=[track_col_intervals, "time_mid_hr", "dI_dt"]).copy()
    df_intervals = df_intervals.sort_values([track_col_intervals, "time_mid_hr"]).reset_index(drop=True)

    # union of tracks from both sheets
    track_ids = sorted(
        set(df_points[track_col_points].dropna().tolist()) |
        set(df_intervals[track_col_intervals].dropna().tolist())
    )

    # --------------------------------------------------------
    # Open workbook
    # --------------------------------------------------------
    wb = load_workbook(input_excel_path)

    new_sheet_name = "track_manual_plot_layout"
    if new_sheet_name in wb.sheetnames:
        del wb[new_sheet_name]

    ws = wb.create_sheet(new_sheet_name)

    # --------------------------------------------------------
    # Styles
    # --------------------------------------------------------
    title_font = Font(bold=True, size=14)
    track_font = Font(bold=True, size=12)
    section_font = Font(bold=True, size=11)
    header_font = Font(bold=True)

    track_fill = PatternFill("solid", fgColor="D9EAF7")
    section_fill = PatternFill("solid", fgColor="EAF4E2")
    header_fill = PatternFill("solid", fgColor="F2F2F2")

    center_align = Alignment(horizontal="center")
    left_align = Alignment(horizontal="left")

    # --------------------------------------------------------
    # Header
    # --------------------------------------------------------
    ws["A1"] = "Track manual plot layout"
    ws["A2"] = f"Source file: {input_excel_path.name}"
    ws["A3"] = "Each track contains two sections: point data and interval data"

    ws["A1"].font = title_font
    ws["A2"].font = Font(italic=True)
    ws["A3"].font = Font(italic=True)

    # column widths
    ws.column_dimensions["A"].width = 20
    ws.column_dimensions["B"].width = 16
    ws.column_dimensions["C"].width = 20
    ws.column_dimensions["D"].width = 16

    current_row = 5

    # --------------------------------------------------------
    # Write one block per track
    # --------------------------------------------------------
    for track_id in track_ids:
        track_points = df_points[df_points[track_col_points] == track_id].copy()
        track_intervals = df_intervals[df_intervals[track_col_intervals] == track_id].copy()

        # track title
        ws.cell(row=current_row, column=1, value=f"track ID {track_id}")
        ws.cell(row=current_row, column=1).font = track_font
        ws.cell(row=current_row, column=1).fill = track_fill

        current_row += 1

        # ----------------------------
        # point data section
        # ----------------------------
        ws.cell(row=current_row, column=1, value="point data")
        ws.cell(row=current_row, column=1).font = section_font
        ws.cell(row=current_row, column=1).fill = section_fill

        current_row += 1

        ws.cell(row=current_row, column=1, value="time_min")
        ws.cell(row=current_row, column=2, value="mean_r")

        ws.cell(row=current_row, column=1).font = header_font
        ws.cell(row=current_row, column=2).font = header_font
        ws.cell(row=current_row, column=1).fill = header_fill
        ws.cell(row=current_row, column=2).fill = header_fill

        current_row += 1

        if len(track_points) > 0:
            for _, row in track_points.iterrows():
                ws.cell(row=current_row, column=1, value=float(row["time_min"]))
                ws.cell(row=current_row, column=2, value=float(row["mean_r"]))
                current_row += 1
        else:
            ws.cell(row=current_row, column=1, value="(no point data)")
            current_row += 1

        current_row += 1

        # ----------------------------
        # interval data section
        # ----------------------------
        ws.cell(row=current_row, column=1, value="interval data")
        ws.cell(row=current_row, column=1).font = section_font
        ws.cell(row=current_row, column=1).fill = section_fill

        current_row += 1

        ws.cell(row=current_row, column=1, value="time_mid_hr")
        ws.cell(row=current_row, column=2, value="dI_dt")

        ws.cell(row=current_row, column=1).font = header_font
        ws.cell(row=current_row, column=2).font = header_font
        ws.cell(row=current_row, column=1).fill = header_fill
        ws.cell(row=current_row, column=2).fill = header_fill

        current_row += 1

        if len(track_intervals) > 0:
            for _, row in track_intervals.iterrows():
                ws.cell(row=current_row, column=1, value=float(row["time_mid_hr"]))
                ws.cell(row=current_row, column=2, value=float(row["dI_dt"]))
                current_row += 1
        else:
            ws.cell(row=current_row, column=1, value="(no interval data)")
            current_row += 1

        # blank gap between tracks
        current_row += 2

    wb.save(output_excel_path)

    return {
        "input_excel": str(input_excel_path),
        "output_excel": str(output_excel_path),
        "n_tracks_points": int(df_points[track_col_points].nunique()),
        "n_tracks_intervals": int(df_intervals[track_col_intervals].nunique()),
        "status": "ok",
    }

In [23]:
from pathlib import Path
import pandas as pd

# ============================================================
# User settings
# ============================================================
BASE_OUTPUT_DIR = Path("outputs") / "individual_blob_kinetics"

# เช่น "WT replicate 3"
# ถ้าจะรันทั้งหมด ใช้ None
REPLICATE_FILTER = "WT replicate 3"

# ถ้า False จะไม่เขียนทับ output เดิม
OVERWRITE_OUTPUT = True


# ============================================================
# Find kinetics Excel files
# ============================================================
excel_files = sorted(BASE_OUTPUT_DIR.rglob("*_individual_blob_kinetics.xlsx"))

# exclude derived outputs
excel_files = [
    f for f in excel_files
    if "_with_track_layout" not in f.name
    and "_with_track_plots" not in f.name
    and "_with_excel_charts" not in f.name
]

if REPLICATE_FILTER is not None:
    excel_files = [
        f for f in excel_files
        if f.parent.name.lower() == REPLICATE_FILTER.lower()
    ]

print("Selected Excel files:")
for f in excel_files:
    print("-", f)

print(f"\nTotal selected files: {len(excel_files)}")


# ============================================================
# Run batch
# ============================================================
batch_results = []
n_total = len(excel_files)

for i, f in enumerate(excel_files, start=1):
    print(f"\n[{i}/{n_total}] Processing: {f.name}")

    try:
        result = add_track_manual_plot_layout_sheet(
            input_excel_path=f,
            output_excel_path=None,
            overwrite_output=OVERWRITE_OUTPUT
        )
        batch_results.append(result)

        print(f"    status             : {result['status']}")
        print(f"    output_excel       : {result['output_excel']}")
        print(f"    n_tracks_points    : {result['n_tracks_points']}")
        print(f"    n_tracks_intervals : {result['n_tracks_intervals']}")

    except Exception as e:
        batch_results.append({
            "input_excel": str(f),
            "output_excel": "",
            "n_tracks_points": None,
            "n_tracks_intervals": None,
            "status": f"error: {e}",
        })
        print(f"    status             : error")
        print(f"    message            : {e}")

batch_results_df = pd.DataFrame(batch_results)

print("\nBatch results summary:")
display(batch_results_df)

Selected Excel files:
- outputs\individual_blob_kinetics\WT replicate 3\WT replicate 3 field 1_individual_blob_kinetics.xlsx
- outputs\individual_blob_kinetics\WT replicate 3\WT replicate 3 field 2_individual_blob_kinetics.xlsx
- outputs\individual_blob_kinetics\WT replicate 3\WT replicate 3 field 3_individual_blob_kinetics.xlsx

Total selected files: 3

[1/3] Processing: WT replicate 3 field 1_individual_blob_kinetics.xlsx
    status             : ok
    output_excel       : outputs\individual_blob_kinetics\WT replicate 3\WT replicate 3 field 1_individual_blob_kinetics_with_track_layout.xlsx
    n_tracks_points    : 22
    n_tracks_intervals : 22

[2/3] Processing: WT replicate 3 field 2_individual_blob_kinetics.xlsx
    status             : ok
    output_excel       : outputs\individual_blob_kinetics\WT replicate 3\WT replicate 3 field 2_individual_blob_kinetics_with_track_layout.xlsx
    n_tracks_points    : 25
    n_tracks_intervals : 25

[3/3] Processing: WT replicate 3 field 3_in

,input_excel,output_excel,n_tracks_points,n_tracks_intervals,status
0,outputs\individual_blob_kinetics\WT replicate ...,outputs\individual_blob_kinetics\WT replicate ...,22,22,ok
1,outputs\individual_blob_kinetics\WT replicate ...,outputs\individual_blob_kinetics\WT replicate ...,25,25,ok
2,outputs\individual_blob_kinetics\WT replicate ...,outputs\individual_blob_kinetics\WT replicate ...,12,12,ok


In [24]:
from pathlib import Path
import shutil
import pandas as pd
import matplotlib.pyplot as plt


def make_track_plot_png(track_df, track_id, save_path, y_min=-0.25, y_max=0.25):
    """
    Create a dI/dt bar plot for one track and save as PNG.
    """
    track_df = track_df.sort_values("time_mid_hr").copy()

    x = track_df["time_mid_hr"].astype(float).values
    y = track_df["dI_dt"].astype(float).values

    colors = ["forestgreen" if val > 0 else "firebrick" for val in y]

    fig, ax = plt.subplots(figsize=(8.5, 3.8))

    if len(x) >= 2:
        diffs = pd.Series(x).diff().dropna()
        min_step = diffs.min() if len(diffs) > 0 else 0.5
        bar_width = max(min_step * 0.35, 0.05)
    else:
        bar_width = 0.08

    ax.bar(
        x,
        y,
        width=bar_width,
        color=colors,
        edgecolor="black",
        linewidth=0.6,
        align="center"
    )

    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.set_ylim(y_min, y_max)

    ax.set_title(f"track ID {track_id}", fontsize=12, pad=10)
    ax.set_xlabel("Time midpoint (hr)", fontsize=10)
    ax.set_ylabel("dI/dt", fontsize=10)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", alpha=0.2, linestyle=":")

    ax.set_xticks(x)
    ax.set_xticklabels([f"{v:.2f}" for v in x], rotation=45, ha="right")
    ax.set_yticks([y_min, 0, y_max])

    offset = 0.015
    for xi, yi in zip(x, y):
        if yi >= 0:
            y_text = min(yi + offset, y_max - 0.005)
            va = "bottom"
        else:
            y_text = max(yi - offset, y_min + 0.005)
            va = "top"

        ax.text(
            xi,
            y_text,
            f"{yi:.2f}",
            ha="center",
            va=va,
            fontsize=8
        )

    plt.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def export_track_plots_png_from_excel(
    input_excel_path,
    output_dir=None,
    overwrite_png=True,
    min_intervals=2,
    y_min=-0.25,
    y_max=0.25
):
    """
    Export one PNG dI/dt plot per track from selected_blob_intervals sheet.

    Parameters
    ----------
    input_excel_path : str or Path
        Existing individual_blob_kinetics Excel file
    output_dir : str or Path or None
        If None, create:
            <input_stem>_track_plots
    overwrite_png : bool
        If True, existing PNGs may be overwritten
    min_intervals : int
        Minimum number of interval rows required to export one track plot
    y_min, y_max : float
        Fixed y-axis range

    Returns
    -------
    dict
    """
    input_excel_path = Path(input_excel_path)

    if output_dir is None:
        output_dir = input_excel_path.parent / f"{input_excel_path.stem}_track_plots"
    else:
        output_dir = Path(output_dir)

    if output_dir.exists() and overwrite_png:
        shutil.rmtree(output_dir, ignore_errors=True)

    output_dir.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------------
    # Read interval data
    # --------------------------------------------------------
    intervals_sheet = "selected_blob_intervals"
    df = pd.read_excel(input_excel_path, sheet_name=intervals_sheet)

    if "track_ID" in df.columns:
        track_col = "track_ID"
    elif "track_id" in df.columns:
        track_col = "track_id"
    else:
        raise ValueError(
            f"{input_excel_path.name}: missing track column "
            f"(expected 'track_ID' or 'track_id')"
        )

    required_cols = [track_col, "time_mid_hr", "dI_dt"]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(
            f"{input_excel_path.name}: missing required columns {missing_cols}"
        )

    df = df.dropna(subset=[track_col, "time_mid_hr", "dI_dt"]).copy()
    df = df.sort_values([track_col, "time_mid_hr"]).reset_index(drop=True)

    plots_exported = 0
    tracks_skipped = 0
    skipped_track_ids = []

    for track_id, track_df in df.groupby(track_col, dropna=True):
        track_df = track_df.sort_values("time_mid_hr").copy()

        if len(track_df) < min_intervals:
            tracks_skipped += 1
            skipped_track_ids.append(track_id)
            continue

        png_path = output_dir / f"track_{track_id}.png"
        make_track_plot_png(
            track_df=track_df,
            track_id=track_id,
            save_path=png_path,
            y_min=y_min,
            y_max=y_max
        )
        plots_exported += 1

    return {
        "input_excel": str(input_excel_path),
        "output_dir": str(output_dir),
        "n_tracks_total": int(df[track_col].nunique()),
        "plots_exported": plots_exported,
        "tracks_skipped": tracks_skipped,
        "skipped_track_ids": skipped_track_ids,
        "status": "ok",
    }

In [25]:
from pathlib import Path
import pandas as pd

# ============================================================
# User settings
# ============================================================
BASE_OUTPUT_DIR = Path("outputs") / "individual_blob_kinetics"

# เช่น "WT replicate 3"
# ถ้าจะรันทั้งหมด ใช้ None
REPLICATE_FILTER = "WT replicate 3"

OVERWRITE_PNG = True
MIN_INTERVALS = 2
Y_MIN = -0.25
Y_MAX = 0.25


# ============================================================
# Find kinetics Excel files
# ============================================================
excel_files = sorted(BASE_OUTPUT_DIR.rglob("*_individual_blob_kinetics.xlsx"))

# exclude derived outputs
excel_files = [
    f for f in excel_files
    if "_with_track_layout" not in f.name
    and "_with_track_plots" not in f.name
    and "_with_excel_charts" not in f.name
]

if REPLICATE_FILTER is not None:
    excel_files = [
        f for f in excel_files
        if f.parent.name.lower() == REPLICATE_FILTER.lower()
    ]

print("Selected Excel files:")
for f in excel_files:
    print("-", f)

print(f"\nTotal selected files: {len(excel_files)}")


# ============================================================
# Run batch
# ============================================================
batch_results = []
n_total = len(excel_files)

for i, f in enumerate(excel_files, start=1):
    print(f"\n[{i}/{n_total}] Processing: {f.name}")

    try:
        result = export_track_plots_png_from_excel(
            input_excel_path=f,
            output_dir=None,
            overwrite_png=OVERWRITE_PNG,
            min_intervals=MIN_INTERVALS,
            y_min=Y_MIN,
            y_max=Y_MAX
        )
        batch_results.append(result)

        print(f"    status          : {result['status']}")
        print(f"    output_dir      : {result['output_dir']}")
        print(f"    n_tracks_total  : {result['n_tracks_total']}")
        print(f"    plots_exported  : {result['plots_exported']}")
        print(f"    tracks_skipped  : {result['tracks_skipped']}")

        if result["tracks_skipped"] > 0:
            print(f"    skipped_track_ids : {result['skipped_track_ids']}")

    except Exception as e:
        batch_results.append({
            "input_excel": str(f),
            "output_dir": "",
            "n_tracks_total": None,
            "plots_exported": None,
            "tracks_skipped": None,
            "skipped_track_ids": None,
            "status": f"error: {e}",
        })
        print(f"    status          : error")
        print(f"    message         : {e}")

batch_results_df = pd.DataFrame(batch_results)

print("\nBatch results summary:")
display(batch_results_df)

Selected Excel files:
- outputs\individual_blob_kinetics\WT replicate 3\WT replicate 3 field 1_individual_blob_kinetics.xlsx
- outputs\individual_blob_kinetics\WT replicate 3\WT replicate 3 field 2_individual_blob_kinetics.xlsx
- outputs\individual_blob_kinetics\WT replicate 3\WT replicate 3 field 3_individual_blob_kinetics.xlsx

Total selected files: 3

[1/3] Processing: WT replicate 3 field 1_individual_blob_kinetics.xlsx
    status          : ok
    output_dir      : outputs\individual_blob_kinetics\WT replicate 3\WT replicate 3 field 1_individual_blob_kinetics_track_plots
    n_tracks_total  : 22
    plots_exported  : 14
    tracks_skipped  : 8
    skipped_track_ids : [3, 16, 17, 18, 19, 20, 21, 22]

[2/3] Processing: WT replicate 3 field 2_individual_blob_kinetics.xlsx
    status          : ok
    output_dir      : outputs\individual_blob_kinetics\WT replicate 3\WT replicate 3 field 2_individual_blob_kinetics_track_plots
    n_tracks_total  : 25
    plots_exported  : 17
    tracks

,input_excel,output_dir,n_tracks_total,plots_exported,tracks_skipped,skipped_track_ids,status
0,outputs\individual_blob_kinetics\WT replicate ...,outputs\individual_blob_kinetics\WT replicate ...,22,14,8,"[3, 16, 17, 18, 19, 20, 21, 22]",ok
1,outputs\individual_blob_kinetics\WT replicate ...,outputs\individual_blob_kinetics\WT replicate ...,25,17,8,"[10, 17, 19, 21, 22, 23, 24, 25]",ok
2,outputs\individual_blob_kinetics\WT replicate ...,outputs\individual_blob_kinetics\WT replicate ...,12,7,5,"[2, 8, 10, 11, 12]",ok


In [26]:
# ============================================================
# NEW WORKFLOW: track layout + PNG export
# ============================================================

In [27]:
from pathlib import Path
import shutil
import pandas as pd
import matplotlib.pyplot as plt

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment


def get_versioned_file_path(path):
    """
    Return a non-overwriting file path.
    If path exists, create:
      file_v2.xlsx, file_v3.xlsx, ...
    """
    path = Path(path)

    if not path.exists():
        return path

    stem = path.stem
    suffix = path.suffix
    parent = path.parent

    i = 2
    while True:
        candidate = parent / f"{stem}_v{i}{suffix}"
        if not candidate.exists():
            return candidate
        i += 1


def get_versioned_dir_path(path):
    """
    Return a non-overwriting directory path.
    If path exists, create:
      folder_v2, folder_v3, ...
    """
    path = Path(path)

    if not path.exists():
        return path

    parent = path.parent
    name = path.name

    i = 2
    while True:
        candidate = parent / f"{name}_v{i}"
        if not candidate.exists():
            return candidate
        i += 1

In [28]:
def add_track_manual_plot_layout_sheet(
    input_excel_path,
    output_excel_path=None,
    overwrite_output=False
):
    """
    Add a new worksheet named 'track_manual_plot_layout' to an existing
    individual_blob_kinetics Excel file.

    For each track_id, create two sections:
      1) point data    -> time_min | mean_r
      2) interval data -> time_mid_hr | dI_dt

    If overwrite_output=False, output file name is auto-versioned.
    """
    input_excel_path = Path(input_excel_path)

    if output_excel_path is None:
        output_excel_path = input_excel_path.with_name(
            input_excel_path.stem + "_with_track_layout.xlsx"
        )
    else:
        output_excel_path = Path(output_excel_path)

    if not overwrite_output:
        output_excel_path = get_versioned_file_path(output_excel_path)

    # --------------------------------------------------------
    # Read source sheets
    # --------------------------------------------------------
    points_sheet = "selected_blob_points"
    intervals_sheet = "selected_blob_intervals"

    df_points = pd.read_excel(input_excel_path, sheet_name=points_sheet)
    df_intervals = pd.read_excel(input_excel_path, sheet_name=intervals_sheet)

    if "track_ID" in df_points.columns:
        track_col_points = "track_ID"
    elif "track_id" in df_points.columns:
        track_col_points = "track_id"
    else:
        raise ValueError(
            f"{input_excel_path.name}: could not find track column in {points_sheet}"
        )

    if "track_ID" in df_intervals.columns:
        track_col_intervals = "track_ID"
    elif "track_id" in df_intervals.columns:
        track_col_intervals = "track_id"
    else:
        raise ValueError(
            f"{input_excel_path.name}: could not find track column in {intervals_sheet}"
        )

    required_points = [track_col_points, "time_min", "mean_r"]
    required_intervals = [track_col_intervals, "time_mid_hr", "dI_dt"]

    missing_points = [c for c in required_points if c not in df_points.columns]
    missing_intervals = [c for c in required_intervals if c not in df_intervals.columns]

    if missing_points:
        raise ValueError(
            f"{input_excel_path.name}: missing columns in {points_sheet}: {missing_points}"
        )

    if missing_intervals:
        raise ValueError(
            f"{input_excel_path.name}: missing columns in {intervals_sheet}: {missing_intervals}"
        )

    df_points = df_points.dropna(subset=[track_col_points, "time_min", "mean_r"]).copy()
    df_points = df_points.sort_values([track_col_points, "time_min"]).reset_index(drop=True)

    df_intervals = df_intervals.dropna(subset=[track_col_intervals, "time_mid_hr", "dI_dt"]).copy()
    df_intervals = df_intervals.sort_values([track_col_intervals, "time_mid_hr"]).reset_index(drop=True)

    track_ids = sorted(
        set(df_points[track_col_points].dropna().tolist()) |
        set(df_intervals[track_col_intervals].dropna().tolist())
    )

    # --------------------------------------------------------
    # Open workbook
    # --------------------------------------------------------
    wb = load_workbook(input_excel_path)

    new_sheet_name = "track_manual_plot_layout"
    if new_sheet_name in wb.sheetnames:
        del wb[new_sheet_name]

    ws = wb.create_sheet(new_sheet_name)

    # --------------------------------------------------------
    # Styles
    # --------------------------------------------------------
    title_font = Font(bold=True, size=14)
    track_font = Font(bold=True, size=12)
    section_font = Font(bold=True, size=11)
    header_font = Font(bold=True)

    track_fill = PatternFill("solid", fgColor="D9EAF7")
    section_fill = PatternFill("solid", fgColor="EAF4E2")
    header_fill = PatternFill("solid", fgColor="F2F2F2")

    # --------------------------------------------------------
    # Header
    # --------------------------------------------------------
    ws["A1"] = "Track manual plot layout"
    ws["A2"] = f"Source file: {input_excel_path.name}"
    ws["A3"] = "Each track contains two sections: point data and interval data"

    ws["A1"].font = title_font
    ws["A2"].font = Font(italic=True)
    ws["A3"].font = Font(italic=True)

    ws.column_dimensions["A"].width = 20
    ws.column_dimensions["B"].width = 16
    ws.column_dimensions["C"].width = 20
    ws.column_dimensions["D"].width = 16

    current_row = 5

    # --------------------------------------------------------
    # Write one block per track
    # --------------------------------------------------------
    for track_id in track_ids:
        track_points = df_points[df_points[track_col_points] == track_id].copy()
        track_intervals = df_intervals[df_intervals[track_col_intervals] == track_id].copy()

        ws.cell(row=current_row, column=1, value=f"track ID {track_id}")
        ws.cell(row=current_row, column=1).font = track_font
        ws.cell(row=current_row, column=1).fill = track_fill
        current_row += 1

        # point data
        ws.cell(row=current_row, column=1, value="point data")
        ws.cell(row=current_row, column=1).font = section_font
        ws.cell(row=current_row, column=1).fill = section_fill
        current_row += 1

        ws.cell(row=current_row, column=1, value="time_min")
        ws.cell(row=current_row, column=2, value="mean_r")
        ws.cell(row=current_row, column=1).font = header_font
        ws.cell(row=current_row, column=2).font = header_font
        ws.cell(row=current_row, column=1).fill = header_fill
        ws.cell(row=current_row, column=2).fill = header_fill
        current_row += 1

        if len(track_points) > 0:
            for _, row in track_points.iterrows():
                ws.cell(row=current_row, column=1, value=float(row["time_min"]))
                ws.cell(row=current_row, column=2, value=float(row["mean_r"]))
                current_row += 1
        else:
            ws.cell(row=current_row, column=1, value="(no point data)")
            current_row += 1

        current_row += 1

        # interval data
        ws.cell(row=current_row, column=1, value="interval data")
        ws.cell(row=current_row, column=1).font = section_font
        ws.cell(row=current_row, column=1).fill = section_fill
        current_row += 1

        ws.cell(row=current_row, column=1, value="time_mid_hr")
        ws.cell(row=current_row, column=2, value="dI_dt")
        ws.cell(row=current_row, column=1).font = header_font
        ws.cell(row=current_row, column=2).font = header_font
        ws.cell(row=current_row, column=1).fill = header_fill
        ws.cell(row=current_row, column=2).fill = header_fill
        current_row += 1

        if len(track_intervals) > 0:
            for _, row in track_intervals.iterrows():
                ws.cell(row=current_row, column=1, value=float(row["time_mid_hr"]))
                ws.cell(row=current_row, column=2, value=float(row["dI_dt"]))
                current_row += 1
        else:
            ws.cell(row=current_row, column=1, value="(no interval data)")
            current_row += 1

        current_row += 2

    wb.save(output_excel_path)

    return {
        "input_excel": str(input_excel_path),
        "output_excel": str(output_excel_path),
        "n_tracks_points": int(df_points[track_col_points].nunique()),
        "n_tracks_intervals": int(df_intervals[track_col_intervals].nunique()),
        "status": "ok",
    }

In [29]:
def make_track_plot_png(track_df, track_id, save_path, y_min=-0.25, y_max=0.25):
    """
    Create a dI/dt bar plot for one track and save as PNG.
    """
    track_df = track_df.sort_values("time_mid_hr").copy()

    x = track_df["time_mid_hr"].astype(float).values
    y = track_df["dI_dt"].astype(float).values

    colors = ["forestgreen" if val > 0 else "firebrick" for val in y]

    fig, ax = plt.subplots(figsize=(8.5, 3.8))

    if len(x) >= 2:
        diffs = pd.Series(x).diff().dropna()
        min_step = diffs.min() if len(diffs) > 0 else 0.5
        bar_width = max(min_step * 0.35, 0.05)
    else:
        bar_width = 0.08

    ax.bar(
        x,
        y,
        width=bar_width,
        color=colors,
        edgecolor="black",
        linewidth=0.6,
        align="center"
    )

    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.set_ylim(y_min, y_max)

    ax.set_title(f"track ID {track_id}", fontsize=12, pad=10)
    ax.set_xlabel("Time midpoint (hr)", fontsize=10)
    ax.set_ylabel("dI/dt", fontsize=10)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", alpha=0.2, linestyle=":")

    ax.set_xticks(x)
    ax.set_xticklabels([f"{v:.2f}" for v in x], rotation=45, ha="right")
    ax.set_yticks([y_min, 0, y_max])

    offset = 0.015
    for xi, yi in zip(x, y):
        if yi >= 0:
            y_text = min(yi + offset, y_max - 0.005)
            va = "bottom"
        else:
            y_text = max(yi - offset, y_min + 0.005)
            va = "top"

        ax.text(
            xi,
            y_text,
            f"{yi:.2f}",
            ha="center",
            va=va,
            fontsize=8
        )

    plt.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def export_track_plots_png_from_excel(
    input_excel_path,
    output_dir=None,
    overwrite_png=False,
    min_intervals=2,
    y_min=-0.25,
    y_max=0.25
):
    """
    Export one PNG dI/dt plot per track from selected_blob_intervals sheet.

    If overwrite_png=False, output directory name is auto-versioned.
    """
    input_excel_path = Path(input_excel_path)

    if output_dir is None:
        output_dir = input_excel_path.parent / f"{input_excel_path.stem}_track_plots"
    else:
        output_dir = Path(output_dir)

    if overwrite_png:
        if output_dir.exists():
            shutil.rmtree(output_dir, ignore_errors=True)
        output_dir.mkdir(parents=True, exist_ok=True)
    else:
        output_dir = get_versioned_dir_path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

    intervals_sheet = "selected_blob_intervals"
    df = pd.read_excel(input_excel_path, sheet_name=intervals_sheet)

    if "track_ID" in df.columns:
        track_col = "track_ID"
    elif "track_id" in df.columns:
        track_col = "track_id"
    else:
        raise ValueError(
            f"{input_excel_path.name}: missing track column "
            f"(expected 'track_ID' or 'track_id')"
        )

    required_cols = [track_col, "time_mid_hr", "dI_dt"]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(
            f"{input_excel_path.name}: missing required columns {missing_cols}"
        )

    df = df.dropna(subset=[track_col, "time_mid_hr", "dI_dt"]).copy()
    df = df.sort_values([track_col, "time_mid_hr"]).reset_index(drop=True)

    plots_exported = 0
    tracks_skipped = 0
    skipped_track_ids = []

    for track_id, track_df in df.groupby(track_col, dropna=True):
        track_df = track_df.sort_values("time_mid_hr").copy()

        if len(track_df) < min_intervals:
            tracks_skipped += 1
            skipped_track_ids.append(track_id)
            continue

        png_path = output_dir / f"track_{track_id}.png"
        make_track_plot_png(
            track_df=track_df,
            track_id=track_id,
            save_path=png_path,
            y_min=y_min,
            y_max=y_max
        )
        plots_exported += 1

    return {
        "input_excel": str(input_excel_path),
        "output_dir": str(output_dir),
        "n_tracks_total": int(df[track_col].nunique()),
        "plots_exported": plots_exported,
        "tracks_skipped": tracks_skipped,
        "skipped_track_ids": skipped_track_ids,
        "status": "ok",
    }

In [41]:
# ============================================================
# Notebook starter controls
# ============================================================
BASE_OUTPUT_DIR = Path("outputs") / "individual_blob_kinetics"

# ถ้าจะรันทุก replicate ใช้ None
# ถ้าอยากกรองเฉพาะบาง replicate เช่น "WT replicate 3" ก็ใส่ string ได้
REPLICATE_FILTER = None

# เปิด/ปิด phase
RUN_PHASE_1_LAYOUT = False
RUN_PHASE_2_PNG = True   # ตอนนี้เน้น Phase 1 ก่อน

# overwrite behavior
# False = auto-version outputs, never overwrite existing files/folders
# True  = allow overwrite
OVERWRITE_PHASE_1_EXCEL = False
OVERWRITE_PHASE_2_PNG = False

# PNG plot settings (ยังไม่ใช้ถ้า RUN_PHASE_2_PNG = False)
MIN_INTERVALS = 2
Y_MIN = -0.25
Y_MAX = 0.25


# ============================================================
# Helper: parse replicate name from kinetics filename
# Expected example:
#   WT replicate 3 field 1_individual_blob_kinetics.xlsx
# -> WT replicate 3
# ============================================================
def parse_replicate_name_from_kinetics_filename(filename):
    suffix = "_individual_blob_kinetics.xlsx"
    name = filename

    if name.endswith(suffix):
        name = name[:-len(suffix)]

    if " field " in name:
        return name.split(" field ")[0].strip()

    return "unclassified_replicate"


# ============================================================
# Find kinetics Excel files
# ============================================================
excel_files = sorted(BASE_OUTPUT_DIR.rglob("*_individual_blob_kinetics.xlsx"))

# exclude derived outputs
excel_files = [
    f for f in excel_files
    if "_with_track_layout" not in f.name
    and "_with_track_plots" not in f.name
    and "_with_excel_charts" not in f.name
]

# optional filter by replicate string in filename
if REPLICATE_FILTER is not None:
    excel_files = [
        f for f in excel_files
        if REPLICATE_FILTER.lower() in f.name.lower()
    ]

print("Selected Excel files:")
for f in excel_files:
    rep_name = parse_replicate_name_from_kinetics_filename(f.name)
    print(f"- {f}   | replicate = {rep_name}")

print(f"\nTotal selected files: {len(excel_files)}")


# ============================================================
# Run selected phases
# ============================================================
phase1_results = []
phase2_results = []

n_total = len(excel_files)

for i, f in enumerate(excel_files, start=1):
    replicate_name = parse_replicate_name_from_kinetics_filename(f.name)

    print(f"\n{'='*70}")
    print(f"[{i}/{n_total}] Processing: {f.name}")
    print(f"Replicate: {replicate_name}")
    print(f"{'='*70}")

    # --------------------------------------------------------
    # Phase 1
    # --------------------------------------------------------
    if RUN_PHASE_1_LAYOUT:
        print("Phase 1: add manual-plot layout sheet")

        try:
            replicate_output_dir = BASE_OUTPUT_DIR / replicate_name
            replicate_output_dir.mkdir(parents=True, exist_ok=True)

            output_excel_path = replicate_output_dir / (
                f.stem + "_with_track_layout.xlsx"
            )

            result1 = add_track_manual_plot_layout_sheet(
                input_excel_path=f,
                output_excel_path=output_excel_path,
                overwrite_output=OVERWRITE_PHASE_1_EXCEL
            )
            phase1_results.append({
                **result1,
                "replicate_name": replicate_name,
            })

            print(f"    status             : {result1['status']}")
            print(f"    replicate_name     : {replicate_name}")
            print(f"    output_excel       : {result1['output_excel']}")
            print(f"    n_tracks_points    : {result1['n_tracks_points']}")
            print(f"    n_tracks_intervals : {result1['n_tracks_intervals']}")

        except Exception as e:
            phase1_results.append({
                "input_excel": str(f),
                "output_excel": "",
                "replicate_name": replicate_name,
                "n_tracks_points": None,
                "n_tracks_intervals": None,
                "status": f"error: {e}",
            })
            print(f"    status             : error")
            print(f"    message            : {e}")

    # --------------------------------------------------------
    # Phase 2
    # --------------------------------------------------------
    if RUN_PHASE_2_PNG:
        print("Phase 2: export PNG plots")

        try:
            replicate_output_dir = BASE_OUTPUT_DIR / replicate_name
            replicate_output_dir.mkdir(parents=True, exist_ok=True)

            output_dir = replicate_output_dir / f"{f.stem}_track_plots"

            result2 = export_track_plots_png_from_excel(
                input_excel_path=f,
                output_dir=output_dir,
                overwrite_png=OVERWRITE_PHASE_2_PNG,
                min_intervals=MIN_INTERVALS,
                y_min=Y_MIN,
                y_max=Y_MAX
            )
            phase2_results.append({
                **result2,
                "replicate_name": replicate_name,
            })

            print(f"    status            : {result2['status']}")
            print(f"    replicate_name    : {replicate_name}")
            print(f"    output_dir        : {result2['output_dir']}")
            print(f"    n_tracks_total    : {result2['n_tracks_total']}")
            print(f"    plots_exported    : {result2['plots_exported']}")
            print(f"    tracks_skipped    : {result2['tracks_skipped']}")
            if result2["tracks_skipped"] > 0:
                print(f"    skipped_track_ids : {result2['skipped_track_ids']}")

        except Exception as e:
            phase2_results.append({
                "input_excel": str(f),
                "output_dir": "",
                "replicate_name": replicate_name,
                "n_tracks_total": None,
                "plots_exported": None,
                "tracks_skipped": None,
                "skipped_track_ids": None,
                "status": f"error: {e}",
            })
            print(f"    status            : error")
            print(f"    message           : {e}")


# ============================================================
# Summaries
# ============================================================
if RUN_PHASE_1_LAYOUT:
    phase1_results_df = pd.DataFrame(phase1_results)
    print("\nPhase 1 summary:")
    display(phase1_results_df)

if RUN_PHASE_2_PNG:
    phase2_results_df = pd.DataFrame(phase2_results)
    print("\nPhase 2 summary:")
    display(phase2_results_df)

Selected Excel files:
- outputs\individual_blob_kinetics\KO replicate 1\KO replicate 1 field 1_individual_blob_kinetics.xlsx   | replicate = KO replicate 1
- outputs\individual_blob_kinetics\KO replicate 1\KO replicate 1 field 2_individual_blob_kinetics.xlsx   | replicate = KO replicate 1
- outputs\individual_blob_kinetics\KO replicate 1\KO replicate 1 field 3_individual_blob_kinetics.xlsx   | replicate = KO replicate 1
- outputs\individual_blob_kinetics\KO replicate 2\KO replicate 2 field 1_individual_blob_kinetics.xlsx   | replicate = KO replicate 2
- outputs\individual_blob_kinetics\KO replicate 2\KO replicate 2 field 2_individual_blob_kinetics.xlsx   | replicate = KO replicate 2
- outputs\individual_blob_kinetics\KO replicate 2\KO replicate 2 field 3_individual_blob_kinetics.xlsx   | replicate = KO replicate 2
- outputs\individual_blob_kinetics\KO replicate 3\KO replicate 3 field 1_individual_blob_kinetics.xlsx   | replicate = KO replicate 3
- outputs\individual_blob_kinetics\KO re

,input_excel,output_dir,n_tracks_total,plots_exported,tracks_skipped,skipped_track_ids,status,replicate_name
0,outputs\individual_blob_kinetics\KO replicate ...,outputs\individual_blob_kinetics\KO replicate ...,3,1,2,"[1, 3]",ok,KO replicate 1
1,outputs\individual_blob_kinetics\KO replicate ...,outputs\individual_blob_kinetics\KO replicate ...,5,5,0,[],ok,KO replicate 1
2,outputs\individual_blob_kinetics\KO replicate ...,outputs\individual_blob_kinetics\KO replicate ...,5,1,4,"[1, 3, 4, 5]",ok,KO replicate 1
3,outputs\individual_blob_kinetics\KO replicate ...,outputs\individual_blob_kinetics\KO replicate ...,7,2,5,"[1, 3, 5, 6, 7]",ok,KO replicate 2
4,outputs\individual_blob_kinetics\KO replicate ...,outputs\individual_blob_kinetics\KO replicate ...,15,5,10,"[1, 5, 8, 9, 10, 11, 12, 13, 14, 15]",ok,KO replicate 2
5,outputs\individual_blob_kinetics\KO replicate ...,outputs\individual_blob_kinetics\KO replicate ...,8,4,4,"[3, 5, 7, 8]",ok,KO replicate 2
6,outputs\individual_blob_kinetics\KO replicate ...,outputs\individual_blob_kinetics\KO replicate ...,3,1,2,"[1, 3]",ok,KO replicate 3
7,outputs\individual_blob_kinetics\KO replicate ...,outputs\individual_blob_kinetics\KO replicate ...,3,3,0,[],ok,KO replicate 3
8,outputs\individual_blob_kinetics\KO replicate ...,outputs\individual_blob_kinetics\KO replicate ...,13,4,9,"[1, 2, 5, 6, 7, 10, 11, 12, 13]",ok,KO replicate 3
9,outputs\individual_blob_kinetics\WT replicate ...,outputs\individual_blob_kinetics\WT replicate ...,26,18,8,"[15, 16, 18, 22, 23, 24, 25, 26]",ok,WT replicate 1


In [35]:
from pathlib import Path
import os

print("Current working directory:")
print(os.getcwd())

BASE_OUTPUT_DIR = Path("outputs") / "individual_blob_kinetics"
print("\nBASE_OUTPUT_DIR:")
print(BASE_OUTPUT_DIR)
print(BASE_OUTPUT_DIR.resolve())
print("Exists?", BASE_OUTPUT_DIR.exists())

if BASE_OUTPUT_DIR.exists():
    all_xlsx = sorted(BASE_OUTPUT_DIR.rglob("*.xlsx"))
    print(f"\nFound {len(all_xlsx)} .xlsx files total:")
    for f in all_xlsx:
        print("-", f)

    target_files = sorted(BASE_OUTPUT_DIR.rglob("*_individual_blob_kinetics.xlsx"))
    print(f"\nFound {len(target_files)} target kinetics files:")
    for f in target_files:
        print("-", f)

    subdirs = sorted([p for p in BASE_OUTPUT_DIR.iterdir() if p.is_dir()])
    print(f"\nFound {len(subdirs)} subdirectories:")
    for p in subdirs:
        print("-", p.name)
else:
    print("\nBase output directory does not exist.")

Current working directory:
C:\Users\REALOak\OneDrive\Desktop\blobs kinetic analysis

BASE_OUTPUT_DIR:
outputs\individual_blob_kinetics
C:\Users\REALOak\OneDrive\Desktop\blobs kinetic analysis\outputs\individual_blob_kinetics
Exists? True

Found 1 .xlsx files total:
- outputs\individual_blob_kinetics\WT replicate 3 field 1_individual_blob_kinetics.xlsx

Found 1 target kinetics files:
- outputs\individual_blob_kinetics\WT replicate 3 field 1_individual_blob_kinetics.xlsx

Found 0 subdirectories:


In [38]:
from pathlib import Path

INPUT_DIR = Path("inputs")

all_files = sorted([p for p in INPUT_DIR.rglob("*") if p.is_file()])

print(f"Found {len(all_files)} input files:\n")
for f in all_files:
    print(f)

Found 18 input files:

inputs\individual_blob_kinetics\KO replicate 1 field 1.xlsx
inputs\individual_blob_kinetics\KO replicate 1 field 2.xlsx
inputs\individual_blob_kinetics\KO replicate 1 field 3.xlsx
inputs\individual_blob_kinetics\KO replicate 2 field 1.xlsx
inputs\individual_blob_kinetics\KO replicate 2 field 2.xlsx
inputs\individual_blob_kinetics\KO replicate 2 field 3.xlsx
inputs\individual_blob_kinetics\KO replicate 3 field 1.xlsx
inputs\individual_blob_kinetics\KO replicate 3 field 2.xlsx
inputs\individual_blob_kinetics\KO replicate 3 field 3.xlsx
inputs\individual_blob_kinetics\WT replicate 1 field 1.xlsx
inputs\individual_blob_kinetics\WT replicate 1 field 2.xlsx
inputs\individual_blob_kinetics\WT replicate 1 field 3.xlsx
inputs\individual_blob_kinetics\WT replicate 2 field 1.xlsx
inputs\individual_blob_kinetics\WT replicate 2 field 2.xlsx
inputs\individual_blob_kinetics\WT replicate 2 field 3.xlsx
inputs\individual_blob_kinetics\WT replicate 3 field 1.xlsx
inputs\individual

In [39]:
from pathlib import Path

INPUT_DIR = Path("inputs")

all_files = sorted([p for p in INPUT_DIR.rglob("*") if p.is_file()])

print("Input filenames only:\n")
for f in all_files:
    print(f.name)

Input filenames only:

KO replicate 1 field 1.xlsx
KO replicate 1 field 2.xlsx
KO replicate 1 field 3.xlsx
KO replicate 2 field 1.xlsx
KO replicate 2 field 2.xlsx
KO replicate 2 field 3.xlsx
KO replicate 3 field 1.xlsx
KO replicate 3 field 2.xlsx
KO replicate 3 field 3.xlsx
WT replicate 1 field 1.xlsx
WT replicate 1 field 2.xlsx
WT replicate 1 field 3.xlsx
WT replicate 2 field 1.xlsx
WT replicate 2 field 2.xlsx
WT replicate 2 field 3.xlsx
WT replicate 3 field 1.xlsx
WT replicate 3 field 2.xlsx
WT replicate 3 field 3.xlsx
